In [39]:
# %%
# %%
# ============================================================
# GPT-5-mini Primary Topic Classification Validation
#
# Chinese AV News Dataset
# Source: Xinhua News Agency
#
# Task:
# Predict dominant topic (topic_primary)
#
# Topic taxonomy is kept identical to the English validation
# to ensure cross-country comparability.
#
# ============================================================


# %%
# If needed:
# pip install openai python-dotenv scikit-learn openpyxl


import os
import time
import json
import pandas as pd
import numpy as np

from pathlib import Path

from dotenv import load_dotenv

from openai import OpenAI


# %%
# %%
# ============================================================
# 0. Load OpenAI API
# ============================================================


load_dotenv(
    "/Users/yurujia/Desktop/Dissertation Data/sentiment/.env"
)


key = os.getenv("OPENAI_API_KEY")


if key:
    print("API key loaded successfully")
else:
    print("API key NOT found")


client = OpenAI()


MODEL = "gpt-5-mini"


# %%
# %%
# ============================================================
# 1. Test API connection
# ============================================================


test_text = """
某自动驾驶企业宣布，其无人驾驶出租车服务在完成新区域道路测试后，
进一步扩大了运营范围。
"""


response = client.responses.create(

    model=MODEL,

    input=f"""

请判断以下自动驾驶新闻文本的主要议题。

只能从以下类别中选择一个：

Technology and Innovation
Safety and Risk
Policy and Regulation
Business and Commercialisation
Public Acceptance and Trust
Mobility and Social Impact
Environment and Sustainability
Legal and Ethics
Other

只返回一个英文类别名称。

新闻文本：

{test_text}

"""

)


print(response.output_text)


# %%
# %%
# ============================================================
# 2. Load Xinhua validation data
# ============================================================


BASE_DIR = Path(
    "/Users/yurujia/Desktop/Dissertation Data/China"
)


XINHUA_PATH = (
    BASE_DIR /
    "excel/"
    "Xinhua_topic_multilabel_annotated_first_av_relevant_paragraph.xlsx"
)


print("Input file:")
print(XINHUA_PATH)


xinhua = pd.read_excel(
    XINHUA_PATH
)


print(
    "\nOriginal shape:",
    xinhua.shape
)


print(
    "\nColumns:"
)

print(
    xinhua.columns.tolist()
)


display(
    xinhua.head()
)


# %%
# %%
# ============================================================
# 3. Specify text and human topic columns
# ============================================================


TEXT_COL = "first_av_relevant_paragraph"

HUMAN_TOPIC_COL = "topic_primary"


assert TEXT_COL in xinhua.columns, (
    f"{TEXT_COL} not found. "
    f"Available columns: {xinhua.columns.tolist()}"
)


assert HUMAN_TOPIC_COL in xinhua.columns, (
    f"{HUMAN_TOPIC_COL} not found. "
    f"Available columns: {xinhua.columns.tolist()}"
)


print(
    "Text column:",
    TEXT_COL
)


print(
    "Human topic column:",
    HUMAN_TOPIC_COL
)


# %%
# %%
# ============================================================
# 4. Standardise validation dataset
# ============================================================


validation = pd.DataFrame({

    "source":
        "Xinhua",

    "text":
        xinhua[TEXT_COL],

    "human_topic":
        xinhua[HUMAN_TOPIC_COL]

})


print(
    "Original validation rows:",
    len(validation)
)


# Remove missing text or human topic labels
validation = validation.dropna(
    subset=[
        "text",
        "human_topic"
    ]
).copy()


# Clean text
validation["text"] = (
    validation["text"]
    .astype(str)
    .str.strip()
)


# Clean human topic labels
validation["human_topic"] = (
    validation["human_topic"]
    .astype(str)
    .str.strip()
)


# Remove empty strings if any
validation = validation[
    validation["text"].ne("")
    &
    validation["human_topic"].ne("")
].reset_index(drop=True)


print(
    "Usable validation rows:",
    len(validation)
)


display(
    validation.head()
)


# %%
# %%
# ============================================================
# 5. Human topic distribution
# ============================================================


print(
    "Human topic distribution:"
)


display(
    validation[
        "human_topic"
    ]
    .value_counts()
)


# %%
# %%
# ============================================================
# 6. Topic categories
#
# IMPORTANT:
# Keep exactly the same English topic labels as the US analysis.
#
# This ensures that:
# 1. Human and GPT labels can be compared directly.
# 2. China-US topic distributions are directly comparable.
# 3. No translation/mapping step is required later.
# ============================================================


TOPICS = [

    "Technology and Innovation",

    "Safety and Risk",

    "Policy and Regulation",

    "Business and Commercialisation",

    "Public Acceptance and Trust",

    "Mobility and Social Impact",

    "Environment and Sustainability",

    "Legal and Ethics",

    "Other"

]


# %%
# %%
# ============================================================
# 7. Validate human topic labels
# ============================================================


unexpected_human_topics = (

    validation.loc[
        ~validation["human_topic"].isin(TOPICS),
        "human_topic"
    ]
    .value_counts()

)


if len(unexpected_human_topics) == 0:

    print(
        "All human topic labels are valid."
    )

else:

    print(
        "WARNING: Unexpected human topic labels found:"
    )

    display(
        unexpected_human_topics
    )


# %%
# %%
# ============================================================
# 8. Topic categories represented in human validation sample
#
# Purpose:
#
# Macro metrics should be calculated across categories that
# actually appear in the manually labelled validation sample.
#
# Example:
# If Environment and Sustainability has 0 human-labelled cases,
# it cannot meaningfully contribute to validation Macro F1.
#
# This is identical to the English validation logic.
# ============================================================


observed_topics = [

    topic

    for topic in TOPICS

    if topic in validation[
        "human_topic"
    ].dropna().unique()

]


print(
    "Observed topics in human validation sample:"
)


for topic in observed_topics:

    print(
        "-",
        topic
    )


print(
    "\nNumber of observed topic categories:",
    len(observed_topics)
)


# %%
# %%
# ============================================================
# 9. P1-CN
# Basic zero-shot primary topic prompt
# ============================================================


def build_prompt_p1_cn(text):

    return f"""

你正在为一项关于自动驾驶新闻报道的学术研究进行主题分类。

请识别以下新闻文本中占主导地位的单一主要议题。

必须从以下类别中选择且只能选择一个：

Technology and Innovation

Safety and Risk

Policy and Regulation

Business and Commercialisation

Public Acceptance and Trust

Mobility and Social Impact

Environment and Sustainability

Legal and Ethics

Other


不要创建新的类别。

只返回一个上述英文类别名称，不要解释。


新闻文本：

{text}

""".strip()


# %%
# %%
# ============================================================
# 10. P2-CN
# Definition-based zero-shot primary topic prompt
# ============================================================


def build_prompt_p2_cn(text):

    return f"""

你是一名专门研究自动驾驶汽车新闻报道的媒体分析专家。

你的任务是识别以下自动驾驶相关新闻文本的
单一主导议题（primary frame）。

必须从下面预先定义的类别中选择且只能选择一个。

不要创建新的类别。
不要使用其他表达方式。
最终只返回英文类别名称。


可选议题：


1. Technology and Innovation

当新闻主要关注自动驾驶技术本身的研发、改进、测试
或技术能力时，选择该类别。

包括：

- 人工智能系统
- 传感器、软件、算法
- 车辆技术系统
- 工程研发
- 技术展示
- 自动驾驶能力提升
- 技术测试和验证

重要：

文章中出现企业、产品或自动驾驶部署，
并不意味着一定属于 Business。

当文章的核心问题是：

“自动驾驶技术如何运行、发展或得到改进？”

应选择 Technology and Innovation。


--------------------------------------------------


2. Safety and Risk

当新闻主要关注自动驾驶汽车的风险、故障、事故、
可靠性或安全评估时，选择该类别。

包括：

- 交通事故
- 碰撞
- 调查
- 运行故障
- 安全担忧
- 可靠性问题
- 风险降低措施

重要：

即使文章同时提到公众反应，
只要核心内容是安全问题，
仍然应选择 Safety and Risk。

除非文章主要讨论人们的态度、
信任或使用意愿，
否则不要选择 Public Acceptance and Trust。


--------------------------------------------------


3. Policy and Regulation

当新闻主要关注政府行动、规则或监管框架时，
选择该类别。

包括：

- 法规或立法
- 政府政策
- 监管批准
- 测试许可
- 官方标准
- 政府或公共机构的决定

当文章的核心问题是：

“自动驾驶如何受到政府治理或监管？”

应选择 Policy and Regulation。


--------------------------------------------------


4. Business and Commercialisation

当新闻主要关注与自动驾驶有关的经济、
企业或市场活动时，选择该类别。

包括：

- 企业战略
- 投资
- 收购
- 企业合作
- 企业竞争
- 财务表现
- 商业模式
- 产业或市场发展

重要：

不要仅仅因为以下情况就选择 Business：

- 文本提到某家公司；
- 企业进行了自动驾驶测试；
- 某项自动驾驶服务正在运营；
- 企业部署了 Robotaxi。

只有当新闻的核心问题是：

“企业、市场或产业如何发展自动驾驶相关业务？”

才应选择 Business and Commercialisation。


--------------------------------------------------


5. Public Acceptance and Trust

当新闻主要关注人们对于自动驾驶汽车的态度、
观点、信任、担忧或使用意愿时，
选择该类别。

包括：

- 消费者接受度
- 公众意见
- 对自动驾驶技术的信任
- 对采用自动驾驶的恐惧或犹豫
- 社会认知

重要：

安全事故本身并不自动属于 Public Acceptance。

如果新闻重点是技术或运行安全风险，
应选择 Safety and Risk。


--------------------------------------------------


6. Mobility and Social Impact

当新闻主要关注自动驾驶作为交通出行服务，
或者其对交通和社会产生的更广泛影响时，
选择该类别。

包括：

- Robotaxi 服务
- 自动驾驶网约车
- 自动驾驶载客运输
- 出行便利性
- 交通系统变化
- 城市交通
- 对日常生活的影响
- 对社会的影响

重要：

当文章重点讨论 Robotaxi、
自动驾驶出租车或自动驾驶载客服务本身时，
通常应选择 Mobility and Social Impact。

只有当文章主要关注企业战略、
投资、竞争或商业市场时，
才选择 Business and Commercialisation。


--------------------------------------------------


7. Environment and Sustainability

当新闻主要关注自动驾驶的环境影响时，
选择该类别。

包括：

- 减少排放
- 能源效率
- 可持续发展
- 环境效益
- 环境问题


--------------------------------------------------


8. Legal and Ethics

当新闻主要关注法律责任或伦理问题时，
选择该类别。

包括：

- 法律责任
- 事故后的责任认定
- 法律纠纷
- 伦理困境
- 问责问题


--------------------------------------------------


主要议题判断规则：


1.

选择最能代表新闻核心框架的议题，
而不是把文章中所有出现过的议题都考虑进去。


2.

不要根据孤立的关键词进行分类。


3.

当多个议题同时出现时，
判断哪个议题在文本中得到最大的强调。


4.

特别区分：

- 技术研发与技术能力
  → Technology and Innovation

- 企业战略、投资或市场活动
  → Business and Commercialisation

- 自动驾驶载客服务和交通出行
  → Mobility and Social Impact

- 政府许可、政策和监管规则
  → Policy and Regulation

- 事故、故障和安全问题
  → Safety and Risk


5.

客观的新闻写作方式本身并不意味着
Business 或 Technology。

必须根据文章实际讨论的核心问题进行分类。


6.

如果自动驾驶仅被简短提及，
而文本不存在明确的自动驾驶主题重点，
选择 Other。


最终只能返回一个英文类别名称。


新闻文本：

{text}

""".strip()


# %%
# %%
# ============================================================
# 11. P3-CN
# Rule-guided zero-shot primary topic prompt
# ============================================================


def build_prompt_p3_cn(text):

    return f"""

你正在对自动驾驶新闻报道的主导框架进行分类。

请选择且只能选择一个主要议题。


可选议题：

Technology and Innovation

Safety and Risk

Policy and Regulation

Business and Commercialisation

Public Acceptance and Trust

Mobility and Social Impact

Environment and Sustainability

Legal and Ethics

Other


请按照以下规则判断：


1.

选择最能代表整段新闻核心框架的议题，
而不是次要议题。


2.

不要根据企业名称、单个关键词或孤立提及进行分类。


3.

仅仅出现企业或公司，
并不意味着文章属于
Business and Commercialisation。


4.

特别区分：

- 技术研发、工程进展、测试能力
  → Technology and Innovation

- 企业战略、投资、竞争或市场活动
  → Business and Commercialisation


5.

当 Robotaxi、自动驾驶网约车、
自动驾驶出租车或其他自动驾驶载客服务
主要被作为交通服务或出行方式讨论时，

通常应选择：

Mobility and Social Impact。


6.

只有当文章主要关注公众态度、信任、
担忧或采用自动驾驶的意愿时，

才选择：

Public Acceptance and Trust。


7.

文本中出现怀疑、不确定性或担忧，
并不自动意味着 Public Acceptance。

如果文章主要讨论交通模式变化、
未来交通或社会出行，

应选择：

Mobility and Social Impact。


8.

事故、碰撞、故障、安全问题或可靠性问题，

应选择：

Safety and Risk，

即使文章同时提到公众反应。


9.

政府批准、测试许可、法律法规、
政策和官方标准，

应选择：

Policy and Regulation。


10.

必须根据占主导地位的新闻框架，
选择且只能选择一个类别。


不要创建新的类别。

最终只返回一个上述英文类别名称。


新闻文本：

{text}

""".strip()


# %%
# %%
# ============================================================
# 12. API classification function
# ============================================================


def classify_topic(prompt, max_retries=3):

    for attempt in range(max_retries):

        try:

            response = client.responses.create(

                model=MODEL,

                input=prompt

            )


            result = (
                response.output_text
                .strip()
            )


            # -----------------------------------------------
            # Normal expected output
            # -----------------------------------------------

            if result in TOPICS:

                return result


            # -----------------------------------------------
            # Light cleaning for slightly malformed output
            # -----------------------------------------------

            cleaned = (
                result
                .replace("Topic:", "")
                .replace("Topic：", "")
                .replace("主题:", "")
                .replace("主题：", "")
                .replace("类别:", "")
                .replace("类别：", "")
                .strip()
            )


            # Remove surrounding quotation marks
            cleaned = (
                cleaned
                .strip('"')
                .strip("'")
                .strip()
            )


            if cleaned in TOPICS:

                return cleaned


            print(
                "Unexpected output:",
                repr(result)
            )


            # Return original result so that unexpected labels
            # can be diagnosed later
            return result


        except Exception as e:

            print(
                f"API error on attempt "
                f"{attempt + 1}/{max_retries}:",
                e
            )


            if attempt < max_retries - 1:

                time.sleep(2)

            else:

                return None


# %%
# %%
# ============================================================
# 13. Small test
#
# Test P2-CN on first 5 validation cases
# ============================================================


for idx, row in validation.head(5).iterrows():

    print(
        "=" * 80
    )


    print(
        "Source:",
        row["source"]
    )


    print(
        "Human:",
        row["human_topic"]
    )


    prediction = classify_topic(
        build_prompt_p2_cn(
            row["text"]
        )
    )


    print(
        "GPT:",
        prediction
    )


    print(
        "\nText:"
    )


    print(
        row["text"][:300]
    )



API key loaded successfully
Business and Commercialisation
Input file:
/Users/yurujia/Desktop/Dissertation Data/China/excel/Xinhua_topic_multilabel_annotated_first_av_relevant_paragraph.xlsx

Original shape: (137, 110)

Columns:
['sample_status', 'sample_has_av_relevant_paragraph', 'av_sentiment_auto', 'manual_sentiment', 'manual_sentiment_note', 'sentiment_analysis_text_preview', 'sentiment_analysis_text', 'first_av_relevant_paragraph_preview', 'first_av_relevant_paragraph', 'first_av_relevant_paragraph_before_dateline_removal', 'first_av_relevant_paragraph_found', 'first_av_relevant_paragraph_position', 'first_av_relevant_substantive_position', 'first_av_relevant_paragraph_char_count', 'first_av_relevant_paragraph_chinese_char_count', 'first_av_relevant_paragraph_english_word_count', 'first_av_relevant_paragraph_number_count', 'first_av_relevant_paragraph_approx_text_unit_count', 'av_relevant_xinhua_dateline_removed', 'av_paragraphs_checked_before_match', 'av_nonrelevant_substantive_

,sample_status,sample_has_av_relevant_paragraph,av_sentiment_auto,manual_sentiment,manual_sentiment_note,sentiment_analysis_text_preview,sentiment_analysis_text,first_av_relevant_paragraph_preview,first_av_relevant_paragraph,first_av_relevant_paragraph_before_dateline_removal,...,topic_policy_regulation,topic_business_commercialisation,topic_public_acceptance_trust,topic_mobility_social_impact,topic_environment_sustainability,topic_legal_ethics,topic_other,topic_unclear,topic_primary,topic_label_count
0,retained_from_old_sample,1.0,1.0,1,NaN,王卫东说，中德双方应“共塑创新”，构建双边关系发展新引擎。双方在智能制造、人工智能、自动驾驶...,王卫东说，中德双方应“共塑创新”，构建双边关系发展新引擎。双方在智能制造、人工智能、自动驾驶...,王卫东说，中德双方应“共塑创新”，构建双边关系发展新引擎。双方在智能制造、人工智能、自动驾驶...,王卫东说，中德双方应“共塑创新”，构建双边关系发展新引擎。双方在智能制造、人工智能、自动驾驶...,王卫东说，中德双方应“共塑创新”，构建双边关系发展新引擎。双方在智能制造、人工智能、自动驾驶...,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,Business and Commercialisation,2.0
1,retained_from_old_sample,1.0,1.0,1,NaN,此次建成后的创新中心，将容纳此前的吉利欧洲研发中心（ＣＥＶＴ）、动力系统研发中心、吉利设计造...,此次建成后的创新中心，将容纳此前的吉利欧洲研发中心（ＣＥＶＴ）、动力系统研发中心、吉利设计造...,此次建成后的创新中心，将容纳此前的吉利欧洲研发中心（ＣＥＶＴ）、动力系统研发中心、吉利设计造...,此次建成后的创新中心，将容纳此前的吉利欧洲研发中心（ＣＥＶＴ）、动力系统研发中心、吉利设计造...,此次建成后的创新中心，将容纳此前的吉利欧洲研发中心（ＣＥＶＴ）、动力系统研发中心、吉利设计造...,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,Technology and Innovation,2.0
2,retained_from_old_sample,1.0,0.0,0,NaN,无人驾驶的汽车如何应对现实环境中可能出现的各种问题？２８日，６３支车队齐聚天津拼比“智能”，...,无人驾驶的汽车如何应对现实环境中可能出现的各种问题？２８日，６３支车队齐聚天津拼比“智能”，...,无人驾驶的汽车如何应对现实环境中可能出现的各种问题？２８日，６３支车队齐聚天津拼比“智能”，...,无人驾驶的汽车如何应对现实环境中可能出现的各种问题？２８日，６３支车队齐聚天津拼比“智能”，...,新华社天津６月２８日电（记者李鲲、钟群）无人驾驶的汽车如何应对现实环境中可能出现的各种问题？...,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Technology and Innovation,2.0
3,retained_from_old_sample,1.0,1.0,1,NaN,智能出行和无人驾驶技术的交流合作，是斯奈德此次访华的重点内容。他说，七年前首次访华时，还少有...,智能出行和无人驾驶技术的交流合作，是斯奈德此次访华的重点内容。他说，七年前首次访华时，还少有...,智能出行和无人驾驶技术的交流合作，是斯奈德此次访华的重点内容。他说，七年前首次访华时，还少有...,智能出行和无人驾驶技术的交流合作，是斯奈德此次访华的重点内容。他说，七年前首次访华时，还少有...,智能出行和无人驾驶技术的交流合作，是斯奈德此次访华的重点内容。他说，七年前首次访华时，还少有...,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,Business and Commercialisation,2.0
4,retained_from_old_sample,1.0,1.0,1,NaN,自动停车是通往无人驾驶道路上重要的里程碑。多家国际知名车企、供应商都在积极研发自动停车系统，...,自动停车是通往无人驾驶道路上重要的里程碑。多家国际知名车企、供应商都在积极研发自动停车系统，...,自动停车是通往无人驾驶道路上重要的里程碑。多家国际知名车企、供应商都在积极研发自动停车系统，...,自动停车是通往无人驾驶道路上重要的里程碑。多家国际知名车企、供应商都在积极研发自动停车系统，...,自动停车是通往无人驾驶道路上重要的里程碑。多家国际知名车企、供应商都在积极研发自动停车系统，...,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Technology and Innovation,2.0


Text column: first_av_relevant_paragraph
Human topic column: topic_primary
Original validation rows: 137
Usable validation rows: 116


,source,text,human_topic
0,Xinhua,王卫东说，中德双方应“共塑创新”，构建双边关系发展新引擎。双方在智能制造、人工智能、自动驾驶...,Business and Commercialisation
1,Xinhua,此次建成后的创新中心，将容纳此前的吉利欧洲研发中心（ＣＥＶＴ）、动力系统研发中心、吉利设计造...,Technology and Innovation
2,Xinhua,无人驾驶的汽车如何应对现实环境中可能出现的各种问题？２８日，６３支车队齐聚天津拼比“智能”，...,Technology and Innovation
3,Xinhua,智能出行和无人驾驶技术的交流合作，是斯奈德此次访华的重点内容。他说，七年前首次访华时，还少有...,Business and Commercialisation
4,Xinhua,自动停车是通往无人驾驶道路上重要的里程碑。多家国际知名车企、供应商都在积极研发自动停车系统，...,Technology and Innovation


Human topic distribution:


human_topic
Technology and Innovation         54
Business and Commercialisation    22
Policy and Regulation             17
Mobility and Social Impact        13
Public Acceptance and Trust        6
Safety and Risk                    2
Other                              1
Environment and Sustainability     1
Name: count, dtype: int64

All human topic labels are valid.
Observed topics in human validation sample:
- Technology and Innovation
- Safety and Risk
- Policy and Regulation
- Business and Commercialisation
- Public Acceptance and Trust
- Mobility and Social Impact
- Environment and Sustainability
- Other

Number of observed topic categories: 8
Source: Xinhua
Human: Business and Commercialisation
GPT: Business and Commercialisation

Text:
王卫东说，中德双方应“共塑创新”，构建双边关系发展新引擎。双方在智能制造、人工智能、自动驾驶领域签署了多项合作文件，这是两个制造业大国优势互补、互利双赢的强强联合。他还说，中德双方深化务实合作，全方位、宽领域、高水平的特点更加鲜明，双方的合作形式越来越丰富，利益融合越来越深厚。
Source: Xinhua
Human: Technology and Innovation
GPT: Technology and Innovation

Text:
此次建成后的创新中心，将容纳此前的吉利欧洲研发中心（ＣＥＶＴ）、动力系统研发中心、吉利设计造型中心哥德堡工作室、领克欧洲销售和市场团队等。建成后的创新中心面积约７００００－８００００平米，按照高艺术水准、高环保标准建造，主动安全、自动驾驶、互联互通等领域将成为其研究重点。
Source: Xinhua
Human: Technology and Innovation
GPT: Technology and Innovation

Text:
无人驾驶的汽车如何应对现实环境中可能出现的各种问题？２８日，６３支车队齐聚天津拼比“智能”，一场代表中国领先水平的智能驾驶比赛拉开帷幕。
Source: Xinhua
Human: Business and Commercialisation
GPT: Busines

In [40]:

# %%
# %%
# ============================================================
# 14. Run P1-CN / P2-CN / P3-CN
# ============================================================


results = validation.copy()


for col in [

    "P1_CN",

    "P2_CN",

    "P3_CN"

]:

    results[col] = None


for idx, row in results.iterrows():

    print(
        f"Processing "
        f"{idx + 1}/{len(results)}"
    )


    text = row["text"]


    results.at[
        idx,
        "P1_CN"
    ] = classify_topic(
        build_prompt_p1_cn(
            text
        )
    )


    results.at[
        idx,
        "P2_CN"
    ] = classify_topic(
        build_prompt_p2_cn(
            text
        )
    )


    results.at[
        idx,
        "P3_CN"
    ] = classify_topic(
        build_prompt_p3_cn(
            text
        )
    )


print(
    "\nP1-CN / P2-CN / P3-CN completed."
)



Processing 1/116
Processing 2/116
Processing 3/116
Processing 4/116
Processing 5/116
Processing 6/116
Processing 7/116
Processing 8/116
Processing 9/116
Processing 10/116
Processing 11/116
Processing 12/116
Processing 13/116
Processing 14/116
Processing 15/116
Processing 16/116
Processing 17/116
Processing 18/116
Processing 19/116
Processing 20/116
Processing 21/116
Processing 22/116
Processing 23/116
Processing 24/116
Processing 25/116
Processing 26/116
Processing 27/116
Processing 28/116
Processing 29/116
Processing 30/116
Processing 31/116
Processing 32/116
Processing 33/116
Processing 34/116
Processing 35/116
Processing 36/116
Processing 37/116
Processing 38/116
Processing 39/116
Processing 40/116
Processing 41/116
Processing 42/116
Processing 43/116
Processing 44/116
Processing 45/116
Processing 46/116
Processing 47/116
Processing 48/116
Processing 49/116
Processing 50/116
Processing 51/116
Processing 52/116
Processing 53/116
Processing 54/116
Processing 55/116
Processing 56/116
P

In [41]:

# %%
# %%
# ============================================================
# 15. Validate model output labels
# ============================================================


for prompt in [

    "P1_CN",

    "P2_CN",

    "P3_CN"

]:

    unexpected = (

        results.loc[
            results[prompt].notna()
            &
            ~results[prompt].isin(TOPICS),
            prompt
        ]
        .value_counts()

    )


    print(
        "\n" + "=" * 60
    )


    print(
        "Checking:",
        prompt
    )


    if len(unexpected) == 0:

        print(
            "All predictions are valid topic labels."
        )

    else:

        print(
            "WARNING: Unexpected model outputs found:"
        )

        display(
            unexpected
        )





Checking: P1_CN
All predictions are valid topic labels.

Checking: P2_CN
All predictions are valid topic labels.

Checking: P3_CN
All predictions are valid topic labels.


In [42]:
# %%
# %%
# ============================================================
# 16. Evaluation
#
# Main validation metrics:
#
# Accuracy:
# Overall proportion of correct classifications.
#
# Macro Precision / Recall / F1:
# Calculated across topic categories actually represented
# in the human validation sample.
#
# Cohen's Kappa:
# Agreement between GPT and human coding beyond chance.
#
# Same logic as English validation.
# ============================================================


from sklearn.metrics import (

    accuracy_score,

    f1_score,

    precision_score,

    recall_score,

    cohen_kappa_score

)


prompt_cols = [

    "P1_CN",

    "P2_CN",

    "P3_CN"

]


evaluation = []


for p in prompt_cols:


    valid = results[
        results[p].notna()
        &
        results["human_topic"].notna()
        &
        results[p].isin(TOPICS)
    ].copy()


    y_true = valid[
        "human_topic"
    ]


    y_pred = valid[
        p
    ]


    evaluation.append({

        "prompt":
            p,

        "n":
            len(valid),

        "accuracy":
            accuracy_score(
                y_true,
                y_pred
            ),

        "macro_precision":
            precision_score(
                y_true,
                y_pred,
                labels=observed_topics,
                average="macro",
                zero_division=0
            ),

        "macro_recall":
            recall_score(
                y_true,
                y_pred,
                labels=observed_topics,
                average="macro",
                zero_division=0
            ),

        "macro_f1":
            f1_score(
                y_true,
                y_pred,
                labels=observed_topics,
                average="macro",
                zero_division=0
            ),

        "cohen_kappa":
            cohen_kappa_score(
                y_true,
                y_pred
            )

    })


evaluation_df = pd.DataFrame(
    evaluation
)


evaluation_df = evaluation_df[
    [
        "prompt",
        "n",
        "accuracy",
        "macro_f1",
        "macro_precision",
        "macro_recall",
        "cohen_kappa"
    ]
]


print(
    "\nP1-CN / P2-CN / P3-CN evaluation:"
)


display(
    evaluation_df.round(4)
)




P1-CN / P2-CN / P3-CN evaluation:


,prompt,n,accuracy,macro_f1,macro_precision,macro_recall,cohen_kappa
0,P1_CN,116,0.6810,0.4170,0.5014,0.4249,0.5019
1,P2_CN,116,0.7759,0.6247,0.7538,0.5931,0.6702
2,P3_CN,116,0.7672,0.6243,0.7625,0.5857,0.6520


In [43]:
# %%
# %%
# ============================================================
# 16A. Per-topic performance:
# P1-CN / P2-CN / P3-CN
#
# Purpose:
# Compare Precision / Recall / F1 / Support for each topic
# across all three initial prompts.
#
# This is especially important because overall accuracy can
# hide weak performance on low-frequency topic categories.
# ============================================================


from sklearn.metrics import classification_report


# ------------------------------------------------------------
# Helper function
# ------------------------------------------------------------

def get_per_topic_metrics(
    df,
    prompt_col,
    topic_labels
):

    valid = df[
        df[prompt_col].notna()
        &
        df["human_topic"].notna()
        &
        df[prompt_col].isin(TOPICS)
    ].copy()


    y_true = valid[
        "human_topic"
    ]


    y_pred = valid[
        prompt_col
    ]


    report = classification_report(
        y_true,
        y_pred,
        labels=topic_labels,
        target_names=topic_labels,
        output_dict=True,
        zero_division=0
    )


    rows = []


    for topic in topic_labels:

        metrics = report[
            topic
        ]


        rows.append({

            "prompt":
                prompt_col,

            "topic":
                topic,

            "precision":
                metrics[
                    "precision"
                ],

            "recall":
                metrics[
                    "recall"
                ],

            "f1_score":
                metrics[
                    "f1-score"
                ],

            "support":
                int(
                    metrics[
                        "support"
                    ]
                )

        })


    return pd.DataFrame(
        rows
    )


# %%
# ============================================================
# 16B. P1-CN per-topic metrics
# ============================================================


p1_per_topic = get_per_topic_metrics(
    results,
    "P1_CN",
    TOPICS
)


print(
    "\nP1-CN per-topic performance:"
)


display(
    p1_per_topic.round(4)
)


# %%
# ============================================================
# 16C. P2-CN per-topic metrics
# ============================================================


p2_per_topic = get_per_topic_metrics(
    results,
    "P2_CN",
    TOPICS
)


print(
    "\nP2-CN per-topic performance:"
)


display(
    p2_per_topic.round(4)
)


# %%
# ============================================================
# 16D. P3-CN per-topic metrics
# ============================================================


p3_per_topic = get_per_topic_metrics(
    results,
    "P3_CN",
    TOPICS
)


print(
    "\nP3-CN per-topic performance:"
)


display(
    p3_per_topic.round(4)
)


# %%
# ============================================================
# 16E. Combine P1 / P2 / P3 per-topic performance
# ============================================================


per_topic_initial_comparison = pd.concat(
    [
        p1_per_topic,
        p2_per_topic,
        p3_per_topic
    ],
    ignore_index=True
)


print(
    "\nCombined per-topic comparison:"
)


display(
    per_topic_initial_comparison.round(4)
)


# %%
# ============================================================
# 16F. Human topic support
#
# Important:
# Low-support categories can have unstable F1 scores.
# ============================================================


topic_support = (
    results[
        "human_topic"
    ]
    .value_counts()
    .reindex(
        TOPICS,
        fill_value=0
    )
    .reset_index()
)


topic_support.columns = [
    "topic",
    "human_support"
]


print(
    "\nHuman topic support:"
)


display(
    topic_support
)


# %%
# ============================================================
# 16G. Direct F1 comparison table
#
# One row per topic.
# Easier to see which prompt performs best for each category.
# ============================================================


p1_f1 = (
    p1_per_topic[
        [
            "topic",
            "f1_score"
        ]
    ]
    .rename(
        columns={
            "f1_score":
                "P1_F1"
        }
    )
)


p2_f1 = (
    p2_per_topic[
        [
            "topic",
            "f1_score"
        ]
    ]
    .rename(
        columns={
            "f1_score":
                "P2_F1"
        }
    )
)


p3_f1 = (
    p3_per_topic[
        [
            "topic",
            "f1_score"
        ]
    ]
    .rename(
        columns={
            "f1_score":
                "P3_F1"
        }
    )
)


f1_initial_comparison = (
    p1_f1
    .merge(
        p2_f1,
        on="topic",
        how="outer"
    )
    .merge(
        p3_f1,
        on="topic",
        how="outer"
    )
)


print(
    "\nPer-topic F1 comparison:"
)


display(
    f1_initial_comparison.round(4)
)


# %%
# ============================================================
# 16H. Identify best initial prompt for each topic
# ============================================================


f1_initial_comparison[
    "best_prompt"
] = (
    f1_initial_comparison[
        [
            "P1_F1",
            "P2_F1",
            "P3_F1"
        ]
    ]
    .idxmax(
        axis=1
    )
)


f1_initial_comparison[
    "best_f1"
] = (
    f1_initial_comparison[
        [
            "P1_F1",
            "P2_F1",
            "P3_F1"
        ]
    ]
    .max(
        axis=1
    )
)


print(
    "\nBest initial prompt by topic:"
)


display(
    f1_initial_comparison.round(4)
)


P1-CN per-topic performance:


,prompt,topic,precision,recall,f1_score,support
0,P1_CN,Technology and Innovation,0.6623,0.9444,0.7786,54
1,P1_CN,Safety and Risk,0.6667,1.0000,0.8000,2
2,P1_CN,Policy and Regulation,1.0000,0.5294,0.6923,17
3,P1_CN,Business and Commercialisation,0.6818,0.6818,0.6818,22
4,P1_CN,Public Acceptance and Trust,0.5000,0.1667,0.2500,6
5,P1_CN,Mobility and Social Impact,0.5000,0.0769,0.1333,13
6,P1_CN,Environment and Sustainability,0.0000,0.0000,0.0000,1
7,P1_CN,Legal and Ethics,0.0000,0.0000,0.0000,0
8,P1_CN,Other,0.0000,0.0000,0.0000,1



P2-CN per-topic performance:


,prompt,topic,precision,recall,f1_score,support
0,P2_CN,Technology and Innovation,0.7778,0.9074,0.8376,54
1,P2_CN,Safety and Risk,1.0000,0.5000,0.6667,2
2,P2_CN,Policy and Regulation,0.8000,0.7059,0.7500,17
3,P2_CN,Business and Commercialisation,0.8095,0.7727,0.7907,22
4,P2_CN,Public Acceptance and Trust,1.0000,0.1667,0.2857,6
5,P2_CN,Mobility and Social Impact,0.6429,0.6923,0.6667,13
6,P2_CN,Environment and Sustainability,0.0000,0.0000,0.0000,1
7,P2_CN,Legal and Ethics,0.0000,0.0000,0.0000,0
8,P2_CN,Other,1.0000,1.0000,1.0000,1



P3-CN per-topic performance:


...

,topic,human_support
0,Technology and Innovation,54
1,Safety and Risk,2
2,Policy and Regulation,17
3,Business and Commercialisation,22
4,Public Acceptance and Trust,6
5,Mobility and Social Impact,13
6,Environment and Sustainability,1
7,Legal and Ethics,0
8,Other,1



Per-topic F1 comparison:


,topic,P1_F1,P2_F1,P3_F1
0,Business and Commercialisation,0.6818,0.7907,0.6829
1,Environment and Sustainability,0.0000,0.0000,0.0000
2,Legal and Ethics,0.0000,0.0000,0.0000
3,Mobility and Social Impact,0.1333,0.6667,0.7200
4,Other,0.0000,1.0000,1.0000
5,Policy and Regulation,0.6923,0.7500,0.8125
6,Public Acceptance and Trust,0.2500,0.2857,0.2857
7,Safety and Risk,0.8000,0.6667,0.6667
8,Technology and Innovation,0.7786,0.8376,0.8264



Best initial prompt by topic:


,topic,P1_F1,P2_F1,P3_F1,best_prompt,best_f1
0,Business and Commercialisation,0.6818,0.7907,0.6829,P2_F1,0.7907
1,Environment and Sustainability,0.0000,0.0000,0.0000,P1_F1,0.0000
2,Legal and Ethics,0.0000,0.0000,0.0000,P1_F1,0.0000
3,Mobility and Social Impact,0.1333,0.6667,0.7200,P3_F1,0.7200
4,Other,0.0000,1.0000,1.0000,P2_F1,1.0000
5,Policy and Regulation,0.6923,0.7500,0.8125,P3_F1,0.8125
6,Public Acceptance and Trust,0.2500,0.2857,0.2857,P2_F1,0.2857
7,Safety and Risk,0.8000,0.6667,0.6667,P1_F1,0.8000
8,Technology and Innovation,0.7786,0.8376,0.8264,P2_F1,0.8376


In [44]:
# %%
# %%
# ============================================================
# 17. P1-CN raw confusion matrix
#
# P1 currently has the highest Macro F1 among the three
# initial Chinese prompts, so it is treated as the main
# candidate for error analysis.
# ============================================================


from sklearn.metrics import confusion_matrix


valid_p1 = results[
    results["human_topic"].notna()
    &
    results["P1_CN"].notna()
    &
    results["P1_CN"].isin(TOPICS)
].copy()


cm_p1 = confusion_matrix(
    valid_p1[
        "human_topic"
    ],
    valid_p1[
        "P1_CN"
    ],
    labels=TOPICS
)


cm_p1_df = pd.DataFrame(
    cm_p1,
    index=TOPICS,
    columns=TOPICS
)


print(
    "\nP1-CN raw confusion matrix:"
)


display(
    cm_p1_df
)


# %%
# %%
# ============================================================
# 18. P1-CN normalised confusion matrix
#
# Each row represents one human-labelled topic.
# Values show the proportion assigned to each predicted topic.
#
# This makes systematic confusion easier to identify.
# ============================================================


cm_p1_normalised = confusion_matrix(
    valid_p1[
        "human_topic"
    ],
    valid_p1[
        "P1_CN"
    ],
    labels=TOPICS,
    normalize="true"
)


cm_p1_normalised_df = pd.DataFrame(
    cm_p1_normalised,
    index=TOPICS,
    columns=TOPICS
)


print(
    "\nP1-CN normalised confusion matrix:"
)


display(
    cm_p1_normalised_df.round(3)
)


# %%
# %%
# ============================================================
# 19. P1-CN error cases
# ============================================================


p1_errors = results[
    results["P1_CN"].notna()
    &
    (
        results["human_topic"]
        !=
        results["P1_CN"]
    )
].copy()


print(
    "Number of P1-CN errors:",
    len(p1_errors)
)


display(
    p1_errors[
        [
            "source",
            "text",
            "human_topic",
            "P1_CN",
            "P2_CN",
            "P3_CN"
        ]
    ]
)


# %%
# %%
# ============================================================
# 20. P1-CN error transitions
#
# Shows which human topics are systematically being confused
# with which GPT-predicted topics.
# ============================================================


p1_error_transitions = (
    p1_errors[
        p1_errors[
            "P1_CN"
        ].isin(TOPICS)
    ]
    .groupby(
        [
            "human_topic",
            "P1_CN"
        ]
    )
    .size()
    .reset_index(
        name="count"
    )
    .sort_values(
        "count",
        ascending=False
    )
)


print(
    "\nP1-CN error transitions:"
)


display(
    p1_error_transitions
)


P1-CN raw confusion matrix:


,Technology and Innovation,Safety and Risk,Policy and Regulation,Business and Commercialisation,Public Acceptance and Trust,Mobility and Social Impact,Environment and Sustainability,Legal and Ethics,Other
Technology and Innovation,51,1,0,1,0,1,0,0,0
Safety and Risk,0,2,0,0,0,0,0,0,0
Policy and Regulation,6,0,9,2,0,0,0,0,0
Business and Commercialisation,7,0,0,15,0,0,0,0,0
Public Acceptance and Trust,3,0,0,2,1,0,0,0,0
Mobility and Social Impact,10,0,0,1,0,1,1,0,0
Environment and Sustainability,0,0,0,1,0,0,0,0,0
Legal and Ethics,0,0,0,0,0,0,0,0,0
Other,0,0,0,0,1,0,0,0,0



P1-CN normalised confusion matrix:


,Technology and Innovation,Safety and Risk,Policy and Regulation,Business and Commercialisation,Public Acceptance and Trust,Mobility and Social Impact,Environment and Sustainability,Legal and Ethics,Other
Technology and Innovation,0.944,0.019,0.000,0.019,0.000,0.019,0.000,0.0,0.0
Safety and Risk,0.000,1.000,0.000,0.000,0.000,0.000,0.000,0.0,0.0
Policy and Regulation,0.353,0.000,0.529,0.118,0.000,0.000,0.000,0.0,0.0
Business and Commercialisation,0.318,0.000,0.000,0.682,0.000,0.000,0.000,0.0,0.0
Public Acceptance and Trust,0.500,0.000,0.000,0.333,0.167,0.000,0.000,0.0,0.0
Mobility and Social Impact,0.769,0.000,0.000,0.077,0.000,0.077,0.077,0.0,0.0
Environment and Sustainability,0.000,0.000,0.000,1.000,0.000,0.000,0.000,0.0,0.0
Legal and Ethics,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.0,0.0
Other,0.000,0.000,0.000,0.000,1.000,0.000,0.000,0.0,0.0


Number of P1-CN errors: 37


,source,text,human_topic,P1_CN,P2_CN,P3_CN
0,Xinhua,王卫东说，中德双方应“共塑创新”，构建双边关系发展新引擎。双方在智能制造、人工智能、自动驾驶...,Business and Commercialisation,Technology and Innovation,Business and Commercialisation,Technology and Innovation
5,Xinhua,“刷脸”与其他创新相结合，应用前景广阔。结合语音技术和无人驾驶技术，可以自动解锁车辆，以自然...,Mobility and Social Impact,Technology and Innovation,Technology and Innovation,Technology and Innovation
9,Xinhua,正在中国东部水乡乌镇举行的第四届世界互联网大会描绘了一个令人期待的5G时代：早晨醒来，智能家...,Mobility and Social Impact,Technology and Innovation,Technology and Innovation,Mobility and Social Impact
18,Xinhua,一辆自动驾驶出租车２７日驶上日本首都东京的街头，进行载客试运行。,Mobility and Social Impact,Technology and Innovation,Mobility and Social Impact,Mobility and Social Impact
21,Xinhua,通用汽车全球执行副总裁兼通用汽车中国公司总裁钱惠康日前曾表示，通用汽车正积极筹备参展首届进口...,Business and Commercialisation,Technology and Innovation,Technology and Innovation,Technology and Innovation
22,Xinhua,跑完步刷刷脸，就能看到运动数据；坐着无人车，可以逛公园；休息时，可以和亭子“说说话”……来到...,Mobility and Social Impact,Technology and Innovation,Mobility and Social Impact,Mobility and Social Impact
26,Xinhua,环岛旅游公路除了景色优美，还将很有“智慧”。基于5G技术、GPS定位、大数据、物联网等科技手...,Mobility and Social Impact,Technology and Innovation,Technology and Innovation,Technology and Innovation
28,Xinhua,扑面而来的5G（第五代移动通信技术）带来汽车网联化与智能化，中国2018年新能源汽车产销量双...,Technology and Innovation,Mobility and Social Impact,Technology and Innovation,Mobility and Social Impact
33,Xinhua,内塔尼亚胡在与斯洛伐克总理佩莱格里尼会晤时表示，以色列方面同意今年晚些时候与斯方在斯洛伐克举...,Business and Commercialisation,Technology and Innovation,Technology and Innovation,Policy and Regulation
36,Xinhua,2022年杭州亚运会召开时，亚运区域内将全面实现自动驾驶。这是记者从13日第19届亚运会汽车...,Mobility and Social Impact,Technology and Innovation,Mobility and Social Impact,Mobility and Social Impact



P1-CN error transitions:


,human_topic,P1_CN,count
4,Mobility and Social Impact,Technology and Innovation,10
0,Business and Commercialisation,Technology and Innovation,7
7,Policy and Regulation,Technology and Innovation,6
9,Public Acceptance and Trust,Technology and Innovation,3
6,Policy and Regulation,Business and Commercialisation,2
8,Public Acceptance and Trust,Business and Commercialisation,2
1,Environment and Sustainability,Business and Commercialisation,1
2,Mobility and Social Impact,Business and Commercialisation,1
3,Mobility and Social Impact,Environment and Sustainability,1
5,Other,Public Acceptance and Trust,1


In [45]:
# %%
# %%
# ============================================================
# P2-CN diagnostic confusion matrix
#
# P2 has the highest overall Accuracy and Cohen's Kappa,
# but substantially lower Macro F1.
#
# This diagnostic identifies which minority categories are
# responsible for the lower Macro F1.
# ============================================================


valid_p2 = results[
    results["human_topic"].notna()
    &
    results["P2_CN"].notna()
    &
    results["P2_CN"].isin(TOPICS)
].copy()


cm_p2 = confusion_matrix(
    valid_p2[
        "human_topic"
    ],
    valid_p2[
        "P2_CN"
    ],
    labels=TOPICS
)


cm_p2_df = pd.DataFrame(
    cm_p2,
    index=TOPICS,
    columns=TOPICS
)


print(
    "\nP2-CN raw confusion matrix:"
)


display(
    cm_p2_df
)


# Normalised confusion matrix

cm_p2_normalised = confusion_matrix(
    valid_p2[
        "human_topic"
    ],
    valid_p2[
        "P2_CN"
    ],
    labels=TOPICS,
    normalize="true"
)


cm_p2_normalised_df = pd.DataFrame(
    cm_p2_normalised,
    index=TOPICS,
    columns=TOPICS
)


print(
    "\nP2-CN normalised confusion matrix:"
)


display(
    cm_p2_normalised_df.round(3)
)


P2-CN raw confusion matrix:


,Technology and Innovation,Safety and Risk,Policy and Regulation,Business and Commercialisation,Public Acceptance and Trust,Mobility and Social Impact,Environment and Sustainability,Legal and Ethics,Other
Technology and Innovation,49,0,2,2,0,1,0,0,0
Safety and Risk,1,1,0,0,0,0,0,0,0
Policy and Regulation,3,0,12,1,0,1,0,0,0
Business and Commercialisation,4,0,1,17,0,0,0,0,0
Public Acceptance and Trust,2,0,0,0,1,3,0,0,0
Mobility and Social Impact,4,0,0,0,0,9,0,0,0
Environment and Sustainability,0,0,0,1,0,0,0,0,0
Legal and Ethics,0,0,0,0,0,0,0,0,0
Other,0,0,0,0,0,0,0,0,1



P2-CN normalised confusion matrix:


,Technology and Innovation,Safety and Risk,Policy and Regulation,Business and Commercialisation,Public Acceptance and Trust,Mobility and Social Impact,Environment and Sustainability,Legal and Ethics,Other
Technology and Innovation,0.907,0.0,0.037,0.037,0.000,0.019,0.0,0.0,0.0
Safety and Risk,0.500,0.5,0.000,0.000,0.000,0.000,0.0,0.0,0.0
Policy and Regulation,0.176,0.0,0.706,0.059,0.000,0.059,0.0,0.0,0.0
Business and Commercialisation,0.182,0.0,0.045,0.773,0.000,0.000,0.0,0.0,0.0
Public Acceptance and Trust,0.333,0.0,0.000,0.000,0.167,0.500,0.0,0.0,0.0
Mobility and Social Impact,0.308,0.0,0.000,0.000,0.000,0.692,0.0,0.0,0.0
Environment and Sustainability,0.000,0.0,0.000,1.000,0.000,0.000,0.0,0.0,0.0
Legal and Ethics,0.000,0.0,0.000,0.000,0.000,0.000,0.0,0.0,0.0
Other,0.000,0.0,0.000,0.000,0.000,0.000,0.0,0.0,1.0


In [68]:
# %%
# ============================================================
# P2-CN error cases
# ============================================================

p2_errors = results[
    results["P2_CN"].notna()
    &
    (
        results["human_topic"]
        !=
        results["P2_CN"]
    )
].copy()


print(
    "Number of P2-CN errors:",
    len(p2_errors)
)


display(
    p2_errors[
        [
            "source",
            "text",
            "human_topic",
            "P1_CN",
            "P2_CN",
            "P3_CN"
        ]
    ]
)


# %%
# ============================================================
# P2-CN error transitions
# ============================================================

p2_error_transitions = (
    p2_errors[
        p2_errors["P2_CN"].isin(TOPICS)
    ]
    .groupby(
        [
            "human_topic",
            "P2_CN"
        ]
    )
    .size()
    .reset_index(
        name="count"
    )
    .sort_values(
        "count",
        ascending=False
    )
)


print(
    "\nP2-CN error transitions:"
)


display(
    p2_error_transitions
)

Number of P2-CN errors: 26


,source,text,human_topic,P1_CN,P2_CN,P3_CN
5,Xinhua,“刷脸”与其他创新相结合，应用前景广阔。结合语音技术和无人驾驶技术，可以自动解锁车辆，以自然...,Mobility and Social Impact,Technology and Innovation,Technology and Innovation,Technology and Innovation
9,Xinhua,正在中国东部水乡乌镇举行的第四届世界互联网大会描绘了一个令人期待的5G时代：早晨醒来，智能家...,Mobility and Social Impact,Technology and Innovation,Technology and Innovation,Mobility and Social Impact
21,Xinhua,通用汽车全球执行副总裁兼通用汽车中国公司总裁钱惠康日前曾表示，通用汽车正积极筹备参展首届进口...,Business and Commercialisation,Technology and Innovation,Technology and Innovation,Technology and Innovation
25,Xinhua,在自动驾驶领域，大众汽车集团正在中国加速发展自动驾驶技术。目前，奥迪品牌已在北京和无锡两座城...,Technology and Innovation,Technology and Innovation,Policy and Regulation,Technology and Innovation
26,Xinhua,环岛旅游公路除了景色优美，还将很有“智慧”。基于5G技术、GPS定位、大数据、物联网等科技手...,Mobility and Social Impact,Technology and Innovation,Technology and Innovation,Technology and Innovation
32,Xinhua,产业通商资源部表示，韩国吸引外国直接投资的重点领域将包括汽车自动驾驶、医疗、智慧家庭、节能方...,Business and Commercialisation,Business and Commercialisation,Policy and Regulation,Policy and Regulation
33,Xinhua,内塔尼亚胡在与斯洛伐克总理佩莱格里尼会晤时表示，以色列方面同意今年晚些时候与斯方在斯洛伐克举...,Business and Commercialisation,Technology and Innovation,Technology and Innovation,Policy and Regulation
40,Xinhua,王炳南说，与首届进博会相比，第二届进博会新设消费品新品专区，增加养老等题材，新增室外汽车“无...,Public Acceptance and Trust,Business and Commercialisation,Mobility and Social Impact,Mobility and Social Impact
47,Xinhua,自动避让行人或障碍物，虚线变道超车，与前车保持安全车距、处置复杂突发路况……在湖南省长沙市梅...,Technology and Innovation,Technology and Innovation,Mobility and Social Impact,Technology and Innovation
48,Xinhua,物流运输是钢铁企业控疫保产的重要环节。疫情发生以来，宝钢的“智慧物流”在此也显出了奇效。在6...,Business and Commercialisation,Technology and Innovation,Technology and Innovation,Technology and Innovation



P2-CN error transitions:


,human_topic,P2_CN,count
1,Business and Commercialisation,Technology and Innovation,4
3,Mobility and Social Impact,Technology and Innovation,4
6,Policy and Regulation,Technology and Innovation,3
7,Public Acceptance and Trust,Mobility and Social Impact,3
8,Public Acceptance and Trust,Technology and Innovation,2
10,Technology and Innovation,Business and Commercialisation,2
12,Technology and Innovation,Policy and Regulation,2
0,Business and Commercialisation,Policy and Regulation,1
2,Environment and Sustainability,Business and Commercialisation,1
4,Policy and Regulation,Business and Commercialisation,1


In [46]:
# %%
# %%
# ============================================================
# 21. P3-CN raw confusion matrix
# ============================================================


valid_p3 = results[
    results["human_topic"].notna()
    &
    results["P3_CN"].notna()
    &
    results["P3_CN"].isin(TOPICS)
].copy()


cm_p3 = confusion_matrix(
    valid_p3[
        "human_topic"
    ],
    valid_p3[
        "P3_CN"
    ],
    labels=TOPICS
)


cm_p3_df = pd.DataFrame(
    cm_p3,
    index=TOPICS,
    columns=TOPICS
)


print(
    "\nP3-CN raw confusion matrix:"
)


display(
    cm_p3_df
)


# %%
# %%
# ============================================================
# 22. P3-CN normalised confusion matrix
# ============================================================


cm_p3_normalised = confusion_matrix(
    valid_p3[
        "human_topic"
    ],
    valid_p3[
        "P3_CN"
    ],
    labels=TOPICS,
    normalize="true"
)


cm_p3_normalised_df = pd.DataFrame(
    cm_p3_normalised,
    index=TOPICS,
    columns=TOPICS
)


print(
    "\nP3-CN normalised confusion matrix:"
)


display(
    cm_p3_normalised_df.round(3)
)


# %%
# %%
# ============================================================
# 23. P3-CN error cases
# ============================================================


p3_errors = results[
    results["P3_CN"].notna()
    &
    (
        results["human_topic"]
        !=
        results["P3_CN"]
    )
].copy()


print(
    "Number of P3-CN errors:",
    len(p3_errors)
)


display(
    p3_errors[
        [
            "source",
            "text",
            "human_topic",
            "P1_CN",
            "P2_CN",
            "P3_CN"
        ]
    ]
)


# %%
# %%
# ============================================================
# 24. P3-CN error transitions
# ============================================================


p3_error_transitions = (
    p3_errors[
        p3_errors[
            "P3_CN"
        ].isin(TOPICS)
    ]
    .groupby(
        [
            "human_topic",
            "P3_CN"
        ]
    )
    .size()
    .reset_index(
        name="count"
    )
    .sort_values(
        "count",
        ascending=False
    )
)


print(
    "\nP3-CN error transitions:"
)


display(
    p3_error_transitions
)


P3-CN raw confusion matrix:


,Technology and Innovation,Safety and Risk,Policy and Regulation,Business and Commercialisation,Public Acceptance and Trust,Mobility and Social Impact,Environment and Sustainability,Legal and Ethics,Other
Technology and Innovation,50,0,0,3,0,1,0,0,0
Safety and Risk,1,1,0,0,0,0,0,0,0
Policy and Regulation,3,0,13,1,0,0,0,0,0
Business and Commercialisation,6,0,2,14,0,0,0,0,0
Public Acceptance and Trust,3,0,0,0,1,2,0,0,0
Mobility and Social Impact,4,0,0,0,0,9,0,0,0
Environment and Sustainability,0,0,0,1,0,0,0,0,0
Legal and Ethics,0,0,0,0,0,0,0,0,0
Other,0,0,0,0,0,0,0,0,1



P3-CN normalised confusion matrix:


,Technology and Innovation,Safety and Risk,Policy and Regulation,Business and Commercialisation,Public Acceptance and Trust,Mobility and Social Impact,Environment and Sustainability,Legal and Ethics,Other
Technology and Innovation,0.926,0.0,0.000,0.056,0.000,0.019,0.0,0.0,0.0
Safety and Risk,0.500,0.5,0.000,0.000,0.000,0.000,0.0,0.0,0.0
Policy and Regulation,0.176,0.0,0.765,0.059,0.000,0.000,0.0,0.0,0.0
Business and Commercialisation,0.273,0.0,0.091,0.636,0.000,0.000,0.0,0.0,0.0
Public Acceptance and Trust,0.500,0.0,0.000,0.000,0.167,0.333,0.0,0.0,0.0
Mobility and Social Impact,0.308,0.0,0.000,0.000,0.000,0.692,0.0,0.0,0.0
Environment and Sustainability,0.000,0.0,0.000,1.000,0.000,0.000,0.0,0.0,0.0
Legal and Ethics,0.000,0.0,0.000,0.000,0.000,0.000,0.0,0.0,0.0
Other,0.000,0.0,0.000,0.000,0.000,0.000,0.0,0.0,1.0


Number of P3-CN errors: 27


,source,text,human_topic,P1_CN,P2_CN,P3_CN
0,Xinhua,王卫东说，中德双方应“共塑创新”，构建双边关系发展新引擎。双方在智能制造、人工智能、自动驾驶...,Business and Commercialisation,Technology and Innovation,Business and Commercialisation,Technology and Innovation
3,Xinhua,智能出行和无人驾驶技术的交流合作，是斯奈德此次访华的重点内容。他说，七年前首次访华时，还少有...,Business and Commercialisation,Business and Commercialisation,Business and Commercialisation,Technology and Innovation
5,Xinhua,“刷脸”与其他创新相结合，应用前景广阔。结合语音技术和无人驾驶技术，可以自动解锁车辆，以自然...,Mobility and Social Impact,Technology and Innovation,Technology and Innovation,Technology and Innovation
21,Xinhua,通用汽车全球执行副总裁兼通用汽车中国公司总裁钱惠康日前曾表示，通用汽车正积极筹备参展首届进口...,Business and Commercialisation,Technology and Innovation,Technology and Innovation,Technology and Innovation
26,Xinhua,环岛旅游公路除了景色优美，还将很有“智慧”。基于5G技术、GPS定位、大数据、物联网等科技手...,Mobility and Social Impact,Technology and Innovation,Technology and Innovation,Technology and Innovation
28,Xinhua,扑面而来的5G（第五代移动通信技术）带来汽车网联化与智能化，中国2018年新能源汽车产销量双...,Technology and Innovation,Mobility and Social Impact,Technology and Innovation,Mobility and Social Impact
29,Xinhua,当天，广汽传祺发布了由广汽洛杉矶前瞻设计中心主导设计的全新概念汽车ENTRANZE，引发全场...,Technology and Innovation,Technology and Innovation,Technology and Innovation,Business and Commercialisation
32,Xinhua,产业通商资源部表示，韩国吸引外国直接投资的重点领域将包括汽车自动驾驶、医疗、智慧家庭、节能方...,Business and Commercialisation,Business and Commercialisation,Policy and Regulation,Policy and Regulation
33,Xinhua,内塔尼亚胡在与斯洛伐克总理佩莱格里尼会晤时表示，以色列方面同意今年晚些时候与斯方在斯洛伐克举...,Business and Commercialisation,Technology and Innovation,Technology and Innovation,Policy and Regulation
37,Xinhua,美国特斯拉汽车公司22日在其位于加利福尼亚州帕洛阿尔托的总部宣布，预计将于2020年第二季度...,Technology and Innovation,Technology and Innovation,Technology and Innovation,Business and Commercialisation



P3-CN error transitions:


,human_topic,P3_CN,count
1,Business and Commercialisation,Technology and Innovation,6
3,Mobility and Social Impact,Technology and Innovation,4
5,Policy and Regulation,Technology and Innovation,3
7,Public Acceptance and Trust,Technology and Innovation,3
9,Technology and Innovation,Business and Commercialisation,3
0,Business and Commercialisation,Policy and Regulation,2
6,Public Acceptance and Trust,Mobility and Social Impact,2
2,Environment and Sustainability,Business and Commercialisation,1
4,Policy and Regulation,Business and Commercialisation,1
8,Safety and Risk,Technology and Innovation,1


In [47]:


# %%
# %%
# ============================================================
# 19. Save initial P1 / P2 / P3 results
# ============================================================


INITIAL_OUTPUT = (

    BASE_DIR /

    "xinhua_primary_topic_prompt_validation.xlsx"

)


results.to_excel(

    INITIAL_OUTPUT,

    index=False

)


print(
    "Initial validation results saved:"
)


print(
    INITIAL_OUTPUT
)

Initial validation results saved:
/Users/yurujia/Desktop/Dissertation Data/China/xinhua_primary_topic_prompt_validation.xlsx


In [69]:
# %%
# ============================================================
# P2R-CN
# Refined definition-based primary topic prompt
#
# Refinement based on systematic P2-CN error analysis:
#
# 1. Mobility -> Technology
# 2. Business -> Technology
# 3. Policy -> Technology
# 4. Public Acceptance -> Mobility / Technology
#
# The refinement is deliberately limited to observed
# category-boundary problems to reduce overfitting.
# ============================================================


def build_prompt_p2r_cn(text):

    return f"""

你是一名专门研究自动驾驶汽车新闻报道的媒体分析专家。

你的任务是识别以下自动驾驶相关新闻文本的
单一主导议题（primary frame）。

必须从下面预先定义的类别中选择且只能选择一个。

不要创建新的类别。
不要使用其他表达方式。
最终只返回英文类别名称。


可选议题：


1. Technology and Innovation

当新闻主要关注自动驾驶技术本身的研发、改进、测试
或技术能力时，选择该类别。

包括：

- 人工智能系统
- 传感器、软件、算法
- 车辆技术系统
- 工程研发
- 技术展示
- 自动驾驶能力提升
- 技术测试和验证

重要：

只有当文章的核心问题是自动驾驶技术本身如何研发、
运行、测试、改进或提升能力时，
才应优先选择 Technology and Innovation。

如果技术只是用于某项交通服务、商业发展、
政策实施或公众体验的背景，
则不应仅因为出现技术、测试或自动驾驶系统等内容
而选择 Technology and Innovation。


--------------------------------------------------


2. Safety and Risk

当新闻主要关注自动驾驶汽车的风险、故障、事故、
可靠性或安全评估时，选择该类别。

包括：

- 交通事故
- 碰撞
- 调查
- 运行故障
- 安全担忧
- 可靠性问题
- 风险降低措施

如果核心问题是自动驾驶系统的事故、安全表现、
运行风险或可靠性，
选择 Safety and Risk。


--------------------------------------------------


3. Policy and Regulation

当新闻主要关注政府行动、规则或监管框架时，
选择该类别。

包括：

- 法规或立法
- 政府政策
- 监管批准
- 测试许可
- 官方标准
- 政府或公共机构的决定
- 政府设立的自动驾驶测试或示范制度

即使政策涉及自动驾驶技术测试、产业发展或商业化，
只要新闻的核心行为者和核心问题是政府政策、
监管安排、官方许可或制度规则，
应选择 Policy and Regulation。


--------------------------------------------------


4. Business and Commercialisation

当新闻主要关注与自动驾驶有关的经济、
企业或市场活动时，选择该类别。

包括：

- 企业战略
- 投资
- 收购
- 企业合作
- 企业竞争
- 财务表现
- 商业模式
- 商业化推进
- 产业发展
- 市场扩张
- 企业布局

不要仅仅因为企业使用或研发了自动驾驶技术，
就选择 Technology and Innovation。

如果文章核心关注的是企业如何布局自动驾驶业务、
产业如何发展、商业化如何推进、
企业如何投资、合作或竞争，
应选择 Business and Commercialisation。


--------------------------------------------------


5. Public Acceptance and Trust

当新闻主要关注人们对于自动驾驶汽车的态度、
观点、信任、担忧、体验或使用意愿时，
选择该类别。

包括：

- 消费者接受度
- 公众意见
- 信任
- 担忧或犹豫
- 是否愿意使用自动驾驶
- 用户体验与接受程度
- 社会认知

Public Acceptance and Trust 不要求文章必须是正式民调。

如果文章的核心问题是人们如何看待、
体验、接受、信任或担忧自动驾驶，
即使同时讨论 Robotaxi、自动驾驶服务或技术能力，
仍应优先选择 Public Acceptance and Trust。

只有当重点是服务本身如何改变交通出行，
而不是人们对该服务的态度或接受程度时，
才选择 Mobility and Social Impact。


--------------------------------------------------


6. Mobility and Social Impact

当新闻主要关注自动驾驶作为交通出行服务，
或者其对交通和社会产生的更广泛影响时，
选择该类别。

包括：

- Robotaxi 服务
- 自动驾驶出租车
- 自动驾驶网约车
- 自动驾驶载客运输
- 出行便利性
- 城市交通
- 交通系统变化
- 对日常生活或社会的影响

如果自动驾驶技术主要是作为载客交通服务、
Robotaxi 或城市出行解决方案被讨论，
应优先选择 Mobility and Social Impact，
即使文章同时提到技术测试、系统能力或企业。

但是：

如果文章主要讨论人们是否信任、接受或愿意使用该服务，
选择 Public Acceptance and Trust。

如果文章主要讨论企业商业战略或市场扩张，
选择 Business and Commercialisation。


--------------------------------------------------


7. Environment and Sustainability

当新闻主要关注自动驾驶的环境影响时，
选择该类别。

包括：

- 减少排放
- 能源效率
- 可持续发展
- 环境效益
- 环境问题


--------------------------------------------------


8. Legal and Ethics

当新闻主要关注法律责任或伦理问题时，
选择该类别。

包括：

- 法律责任
- 事故后的责任认定
- 法律纠纷
- 伦理困境
- 问责问题


--------------------------------------------------


主要议题判断规则：


1.

选择最能代表新闻核心框架的议题，
而不是把文章中所有出现过的主题都纳入分类。


2.

不要根据孤立关键词进行判断。


3.

当多个主题同时出现时，
先判断文章主要在回答哪一个问题：

- 技术本身如何研发、运行或改进
  → Technology and Innovation

- 企业或产业如何布局、投资、竞争或商业化
  → Business and Commercialisation

- 政府如何许可、规范、推动或限制自动驾驶
  → Policy and Regulation

- 自动驾驶如何作为交通或载客服务被使用
  → Mobility and Social Impact

- 人们是否信任、接受、担忧或愿意使用自动驾驶
  → Public Acceptance and Trust

- 自动驾驶是否安全、可靠或存在事故与风险
  → Safety and Risk


4.

特别注意：

“自动驾驶技术被提及”
不等于 Technology and Innovation。

只有当技术本身是主要分析对象时，
才选择 Technology and Innovation。


5.

如果新闻同时涉及 Robotaxi 或自动驾驶交通服务
以及公众态度：

- 重点是服务运营、交通方式或出行影响
  → Mobility and Social Impact

- 重点是用户态度、信任、担忧、体验或使用意愿
  → Public Acceptance and Trust


6.

如果新闻同时涉及企业和技术：

- 重点是技术研发、性能或能力
  → Technology and Innovation

- 重点是企业战略、产业发展、商业化、投资或市场活动
  → Business and Commercialisation


7.

如果新闻同时涉及政府政策和技术：

- 重点是政策、许可、监管、标准或制度安排
  → Policy and Regulation

- 重点是技术本身的能力、研发或测试表现
  → Technology and Innovation


8.

如果自动驾驶只是被简短提及，
没有形成明确主题重点，
选择 Other。


最终只能返回一个英文类别名称。


新闻文本：

{text}

""".strip()

In [70]:
# %%
# ============================================================
# 21. Test P2-R-CN
# ============================================================

for idx, row in validation.head(5).iterrows():

    print("=" * 80)

    print(
        "Source:",
        row["source"]
    )

    print(
        "Human:",
        row["human_topic"]
    )

    prediction = classify_topic(
        build_prompt_p2r_cn(
            row["text"]
        )
    )

    print(
        "P2-R-CN:",
        prediction
    )


# %%
# ============================================================
# 22. Run P2-R-CN on full validation sample
# ============================================================

results["P2R_CN"] = None


for idx, row in results.iterrows():

    print(
        f"Processing P2-R-CN "
        f"{idx + 1}/{len(results)}"
    )

    results.at[
        idx,
        "P2R_CN"
    ] = classify_topic(
        build_prompt_p2r_cn(
            row["text"]
        )
    )


print(
    "P2-R-CN completed."
)


# %%
# ============================================================
# 23. Validate all model output labels
# ============================================================

for prompt in [

    "P1_CN",
    "P2_CN",
    "P3_CN",
    "P2R_CN"

]:

    if prompt not in results.columns:
        continue

    unexpected = (
        results.loc[
            results[prompt].notna()
            &
            ~results[prompt].isin(TOPICS),
            prompt
        ]
        .value_counts()
    )

    print(
        "\n" + "=" * 60
    )

    print(
        "Checking:",
        prompt
    )

    if len(unexpected) == 0:

        print(
            "All predictions are valid topic labels."
        )

    else:

        print(
            "WARNING: Unexpected model outputs found:"
        )

        display(
            unexpected
        )


# %%
# ============================================================
# 24. Compare P2-CN and P2R-CN
#
# Same evaluation criteria as English analysis.
# ============================================================

prompt_columns = [

    "P2_CN",
    "P2R_CN"

]


comparison_rows = []


for prompt in prompt_columns:

    valid = results[
        results[prompt].notna()
        &
        results["human_topic"].notna()
        &
        results[prompt].isin(TOPICS)
    ].copy()

    y_true = valid[
        "human_topic"
    ]

    y_pred = valid[
        prompt
    ]

    comparison_rows.append({

        "prompt":
            prompt,

        "n":
            len(valid),

        "accuracy":
            accuracy_score(
                y_true,
                y_pred
            ),

        "macro_precision":
            precision_score(
                y_true,
                y_pred,
                labels=observed_topics,
                average="macro",
                zero_division=0
            ),

        "macro_recall":
            recall_score(
                y_true,
                y_pred,
                labels=observed_topics,
                average="macro",
                zero_division=0
            ),

        "macro_f1":
            f1_score(
                y_true,
                y_pred,
                labels=observed_topics,
                average="macro",
                zero_division=0
            ),

        "cohen_kappa":
            cohen_kappa_score(
                y_true,
                y_pred
            )

    })


comparison_df = pd.DataFrame(
    comparison_rows
)


comparison_df = comparison_df[
    [
        "prompt",
        "n",
        "accuracy",
        "macro_f1",
        "macro_precision",
        "macro_recall",
        "cohen_kappa"
    ]
]


print(
    "\nP2-CN vs P2R-CN:"
)


display(
    comparison_df.round(4)
)


# %%
# ============================================================
# 25. Compare all four Chinese prompts
# ============================================================

all_prompt_columns = [

    "P1_CN",
    "P2_CN",
    "P3_CN",
    "P2R_CN"

]


all_evaluation_rows = []


for prompt in all_prompt_columns:

    valid = results[
        results[prompt].notna()
        &
        results["human_topic"].notna()
        &
        results[prompt].isin(TOPICS)
    ].copy()

    y_true = valid[
        "human_topic"
    ]

    y_pred = valid[
        prompt
    ]

    all_evaluation_rows.append({

        "prompt":
            prompt,

        "n":
            len(valid),

        "accuracy":
            accuracy_score(
                y_true,
                y_pred
            ),

        "macro_precision":
            precision_score(
                y_true,
                y_pred,
                labels=observed_topics,
                average="macro",
                zero_division=0
            ),

        "macro_recall":
            recall_score(
                y_true,
                y_pred,
                labels=observed_topics,
                average="macro",
                zero_division=0
            ),

        "macro_f1":
            f1_score(
                y_true,
                y_pred,
                labels=observed_topics,
                average="macro",
                zero_division=0
            ),

        "cohen_kappa":
            cohen_kappa_score(
                y_true,
                y_pred
            )

    })


all_evaluation_df = pd.DataFrame(
    all_evaluation_rows
)


all_evaluation_df = all_evaluation_df[
    [
        "prompt",
        "n",
        "accuracy",
        "macro_f1",
        "macro_precision",
        "macro_recall",
        "cohen_kappa"
    ]
]


print(
    "\nAll Chinese prompt comparison:"
)


display(
    all_evaluation_df.round(4)
)


# %%
# ============================================================
# 26. Confusion matrix P2-R-CN
# ============================================================

valid_p2r = results[
    results["human_topic"].notna()
    &
    results["P2R_CN"].notna()
    &
    results["P2R_CN"].isin(TOPICS)
].copy()


cm_p2r = confusion_matrix(

    valid_p2r[
        "human_topic"
    ],

    valid_p2r[
        "P2R_CN"
    ],

    labels=TOPICS

)


cm_p2r_df = pd.DataFrame(

    cm_p2r,

    index=TOPICS,

    columns=TOPICS

)


print(
    "\nP2R-CN confusion matrix:"
)


display(
    cm_p2r_df
)


# %%
# ============================================================
# 27. P2-R-CN errors
# ============================================================

p2r_errors = results[
    results["P2R_CN"].notna()
    &
    (
        results["human_topic"]
        !=
        results["P2R_CN"]
    )
].copy()


print(
    "Number of P2-R-CN errors:",
    len(p2r_errors)
)


display(

    p2r_errors[
        [
            "source",
            "text",
            "human_topic",
            "P2_CN",
            "P2R_CN"
        ]
    ]

)


# %%
# ============================================================
# 28. Error transitions for P2-R-CN
#
# Shows:
# Human topic -> GPT topic
# ============================================================

p2r_error_transitions = (

    p2r_errors[
        p2r_errors["P2R_CN"].isin(TOPICS)
    ]

    .groupby(
        [
            "human_topic",
            "P2R_CN"
        ]
    )

    .size()

    .reset_index(
        name="count"
    )

    .sort_values(
        "count",
        ascending=False
    )

)


print(
    "\nP2-R-CN error transitions:"
)


display(
    p2r_error_transitions
)


# %%
# ============================================================
# 29. Cases improved / worsened by P2-R-CN
# ============================================================

comparison = results.copy()


comparison["P2_correct"] = (

    comparison["P2_CN"]
    ==
    comparison["human_topic"]

)


comparison["P2R_correct"] = (

    comparison["P2R_CN"]
    ==
    comparison["human_topic"]

)


fixed = comparison[
    (~comparison["P2_correct"])
    &
    (comparison["P2R_correct"])
].copy()


worsened = comparison[
    (comparison["P2_correct"])
    &
    (~comparison["P2R_correct"])
].copy()


print(
    "Fixed by P2-R-CN:",
    len(fixed)
)


print(
    "Worsened by P2-R-CN:",
    len(worsened)
)


print(
    "\nCases fixed by P2-R-CN:"
)


display(
    fixed[
        [
            "text",
            "human_topic",
            "P2_CN",
            "P2R_CN"
        ]
    ]
)


print(
    "\nCases worsened by P2-R-CN:"
)


display(
    worsened[
        [
            "text",
            "human_topic",
            "P2_CN",
            "P2R_CN"
        ]
    ]
)


# %%
# ============================================================
# 30. P2 vs P2-R summary
# ============================================================

p2_vs_p2r_summary = pd.DataFrame({

    "metric": [

        "Cases fixed by P2R",
        "Cases worsened by P2R",
        "Net improvement"

    ],

    "value": [

        len(fixed),
        len(worsened),
        len(fixed) - len(worsened)

    ]

})


display(
    p2_vs_p2r_summary
)


# %%
# ============================================================
# 31. Save P2-R validation results
# ============================================================

OUTPUT = (

    BASE_DIR /

    "xinhua_primary_topic_prompt_validation_with_P2R.xlsx"

)


results.to_excel(

    OUTPUT,

    index=False

)


print(
    "P2-R validation results saved:"
)


print(
    OUTPUT
)


# %%
# ============================================================
# 32. Per-topic classification performance
#
# Purpose:
#
# Examine Precision / Recall / F1 / Support
# for each topic for P2_CN and P2R_CN.
# ============================================================

from sklearn.metrics import classification_report


def get_per_topic_metrics(
    df,
    prompt_col,
    topic_labels
):

    valid = df[
        df[prompt_col].notna()
        &
        df["human_topic"].notna()
        &
        df[prompt_col].isin(TOPICS)
    ].copy()

    y_true = valid[
        "human_topic"
    ]

    y_pred = valid[
        prompt_col
    ]

    report = classification_report(

        y_true,

        y_pred,

        labels=topic_labels,

        target_names=topic_labels,

        output_dict=True,

        zero_division=0

    )

    rows = []

    for topic in topic_labels:

        metrics = report[
            topic
        ]

        rows.append({

            "prompt":
                prompt_col,

            "topic":
                topic,

            "precision":
                metrics[
                    "precision"
                ],

            "recall":
                metrics[
                    "recall"
                ],

            "f1_score":
                metrics[
                    "f1-score"
                ],

            "support":
                int(
                    metrics[
                        "support"
                    ]
                )

        })

    return pd.DataFrame(
        rows
    )


# %%
# ============================================================
# 33. Per-topic metrics for P2-CN
# ============================================================

p2_per_topic = get_per_topic_metrics(

    results,

    "P2_CN",

    TOPICS

)


print(
    "\nP2-CN per-topic metrics:"
)


display(
    p2_per_topic.round(4)
)


# %%
# ============================================================
# 34. Per-topic metrics for P2R-CN
# ============================================================

p2r_per_topic = get_per_topic_metrics(

    results,

    "P2R_CN",

    TOPICS

)


print(
    "\nP2R-CN per-topic metrics:"
)


display(
    p2r_per_topic.round(4)
)


# %%
# ============================================================
# 35. Combine P2 and P2-R per-topic metrics
# ============================================================

per_topic_comparison = pd.concat(

    [

        p2_per_topic,

        p2r_per_topic

    ],

    ignore_index=True

)


display(
    per_topic_comparison.round(4)
)


# %%
# ============================================================
# 36. Human topic support
#
# Important because low-support topics can have unstable F1.
# ============================================================

topic_support = (

    results[
        "human_topic"
    ]

    .value_counts()

    .reindex(
        TOPICS,
        fill_value=0
    )

    .reset_index()

)


topic_support.columns = [

    "topic",

    "human_support"

]


print(
    "\nHuman topic support:"
)


display(
    topic_support
)


# %%
# ============================================================
# 37. Normalised confusion matrix for P2-CN
#
# Each row sums to 1 for categories with observations.
#
# Shows where each human topic is being misclassified.
# ============================================================

valid_p2 = results[
    results["human_topic"].notna()
    &
    results["P2_CN"].notna()
    &
    results["P2_CN"].isin(TOPICS)
].copy()


cm_p2 = confusion_matrix(

    valid_p2[
        "human_topic"
    ],

    valid_p2[
        "P2_CN"
    ],

    labels=TOPICS

)


cm_p2_df = pd.DataFrame(

    cm_p2,

    index=TOPICS,

    columns=TOPICS

)


cm_p2_normalised = confusion_matrix(

    valid_p2[
        "human_topic"
    ],

    valid_p2[
        "P2_CN"
    ],

    labels=TOPICS,

    normalize="true"

)


cm_p2_normalised_df = pd.DataFrame(

    cm_p2_normalised,

    index=TOPICS,

    columns=TOPICS

)


print(
    "\nP2-CN normalised confusion matrix:"
)


display(
    cm_p2_normalised_df.round(3)
)


# %%
# ============================================================
# 38. Normalised confusion matrix for P2R-CN
# ============================================================

cm_p2r_normalised = confusion_matrix(

    valid_p2r[
        "human_topic"
    ],

    valid_p2r[
        "P2R_CN"
    ],

    labels=TOPICS,

    normalize="true"

)


cm_p2r_normalised_df = pd.DataFrame(

    cm_p2r_normalised,

    index=TOPICS,

    columns=TOPICS

)


print(
    "\nP2R-CN normalised confusion matrix:"
)


display(
    cm_p2r_normalised_df.round(3)
)


# %%
# ============================================================
# 39. Identify weak-performing topics
#
# Threshold is diagnostic only.
#
# F1 < 0.50 is NOT treated as a universal statistical cutoff.
# ============================================================

weak_topics_p2 = p2_per_topic[
    p2_per_topic[
        "f1_score"
    ] < 0.50
].copy()


print(
    "P2_CN topics with F1 < 0.50:"
)


display(
    weak_topics_p2.round(4)
)


weak_topics_p2r = p2r_per_topic[
    p2r_per_topic[
        "f1_score"
    ] < 0.50
].copy()


print(
    "\nP2R_CN topics with F1 < 0.50:"
)


display(
    weak_topics_p2r.round(4)
)


# %%
# ============================================================
# 40. Compare per-topic F1 change from P2 to P2-R
# ============================================================

f1_change = (

    p2_per_topic[
        [
            "topic",
            "f1_score"
        ]
    ]

    .rename(
        columns={
            "f1_score":
                "P2_F1"
        }
    )

    .merge(

        p2r_per_topic[
            [
                "topic",
                "f1_score"
            ]
        ]

        .rename(
            columns={
                "f1_score":
                    "P2R_F1"
            }
        ),

        on="topic",

        how="outer"

    )

)


f1_change[
    "F1_change_P2R_minus_P2"
] = (

    f1_change[
        "P2R_F1"
    ]

    -

    f1_change[
        "P2_F1"
    ]

)


print(
    "\nPer-topic F1 change:"
)


display(
    f1_change.round(4)
)


# %%
# ============================================================
# 41. Label distribution comparison
#
# Human vs P1 / P2 / P3 / P2R
#
# Useful for checking whether a prompt systematically
# over-predicts a particular category.
# ============================================================

distribution_columns = [

    "human_topic",

    "P1_CN",

    "P2_CN",

    "P3_CN",

    "P2R_CN"

]


distribution_dict = {}


for col in distribution_columns:

    counts = (

        results[col]

        .dropna()

        .value_counts()

        .reindex(
            TOPICS,
            fill_value=0
        )

    )

    distribution_dict[
        col
    ] = counts


topic_distribution_df = pd.DataFrame(
    distribution_dict
)


print(
    "\nTopic label counts:"
)


display(
    topic_distribution_df
)


# %%
# ============================================================
# 42. Topic distribution percentages
# ============================================================

topic_distribution_pct_df = (

    topic_distribution_df

    /

    topic_distribution_df.sum(
        axis=0
    )

    *

    100

)


print(
    "\nTopic label percentages:"
)


display(
    topic_distribution_pct_df.round(2)
)


# %%
# ============================================================
# 43. Save detailed diagnostic results
# ============================================================

DIAGNOSTIC_OUTPUT = (

    BASE_DIR /

    "xinhua_topic_prompt_detailed_diagnostics_P2R.xlsx"

)


with pd.ExcelWriter(

    DIAGNOSTIC_OUTPUT,

    engine="openpyxl"

) as writer:

    results.to_excel(
        writer,
        sheet_name="article_results",
        index=False
    )

    all_evaluation_df.to_excel(
        writer,
        sheet_name="all_prompt_metrics",
        index=False
    )

    comparison_df.to_excel(
        writer,
        sheet_name="P2_vs_P2R_metrics",
        index=False
    )

    p2_per_topic.to_excel(
        writer,
        sheet_name="P2_per_topic",
        index=False
    )

    p2r_per_topic.to_excel(
        writer,
        sheet_name="P2R_per_topic",
        index=False
    )

    f1_change.to_excel(
        writer,
        sheet_name="F1_comparison",
        index=False
    )

    topic_support.to_excel(
        writer,
        sheet_name="topic_support",
        index=False
    )

    cm_p2_df.to_excel(
        writer,
        sheet_name="P2_confusion_raw"
    )

    cm_p2_normalised_df.to_excel(
        writer,
        sheet_name="P2_confusion_normalised"
    )

    cm_p2r_df.to_excel(
        writer,
        sheet_name="P2R_confusion_raw"
    )

    cm_p2r_normalised_df.to_excel(
        writer,
        sheet_name="P2R_confusion_normalised"
    )

    weak_topics_p2.to_excel(
        writer,
        sheet_name="P2_weak_topics",
        index=False
    )

    weak_topics_p2r.to_excel(
        writer,
        sheet_name="P2R_weak_topics",
        index=False
    )

    p2r_errors.to_excel(
        writer,
        sheet_name="P2R_errors",
        index=False
    )

    p2r_error_transitions.to_excel(
        writer,
        sheet_name="P2R_error_transitions",
        index=False
    )

    fixed.to_excel(
        writer,
        sheet_name="fixed_by_P2R",
        index=False
    )

    worsened.to_excel(
        writer,
        sheet_name="worsened_by_P2R",
        index=False
    )

    p2_vs_p2r_summary.to_excel(
        writer,
        sheet_name="P2_vs_P2R_summary",
        index=False
    )

    topic_distribution_df.to_excel(
        writer,
        sheet_name="topic_counts"
    )

    topic_distribution_pct_df.to_excel(
        writer,
        sheet_name="topic_percentages"
    )


print(
    "Detailed diagnostics saved to:"
)


print(
    DIAGNOSTIC_OUTPUT
)


# %%
# ============================================================
# 44. Supplementary Macro F1 diagnostics
#
# 1. Observed-category Macro F1:
#    Includes all categories represented in human validation,
#    including "Other".
#
# 2. Substantive-topic Macro F1:
#    Excludes the residual "Other" category.
#
# The second measure is supplementary only.
#
# Main dissertation results should still report the
# observed-category Macro F1 used in the main evaluation table.
#
# Same logic as English validation.
# ============================================================

substantive_observed_topics = [

    topic

    for topic in observed_topics

    if topic != "Other"

]


print(
    "Observed categories:"
)


print(
    observed_topics
)


print(
    "\nSubstantive observed categories "
    "(excluding Other):"
)


print(
    substantive_observed_topics
)


# %%
# ============================================================
# 45. Supplementary Macro F1 for P2 and P2R
# ============================================================

supplementary_rows = []


for prompt in [

    "P2_CN",

    "P2R_CN"

]:

    valid = results[
        results[prompt].notna()
        &
        results["human_topic"].notna()
        &
        results[prompt].isin(TOPICS)
    ].copy()

    y_true = valid[
        "human_topic"
    ]

    y_pred = valid[
        prompt
    ]

    observed_macro_f1 = f1_score(

        y_true,

        y_pred,

        labels=observed_topics,

        average="macro",

        zero_division=0

    )

    substantive_macro_f1 = f1_score(

        y_true,

        y_pred,

        labels=substantive_observed_topics,

        average="macro",

        zero_division=0

    )

    supplementary_rows.append({

        "prompt":
            prompt,

        "observed_category_macro_f1":
            observed_macro_f1,

        "substantive_topic_macro_f1_excluding_other":
            substantive_macro_f1,

        "number_observed_categories":
            len(
                observed_topics
            ),

        "number_substantive_categories":
            len(
                substantive_observed_topics
            )

    })


supplementary_f1_df = pd.DataFrame(
    supplementary_rows
)


print(
    "\nSupplementary Macro F1 diagnostics:"
)


display(
    supplementary_f1_df.round(4)
)


# %%
# ============================================================
# 46. Save supplementary Macro F1 diagnostics
# ============================================================

SUPPLEMENTARY_OUTPUT = (

    BASE_DIR /

    "xinhua_topic_prompt_supplementary_macro_f1_P2R.xlsx"

)


supplementary_f1_df.to_excel(

    SUPPLEMENTARY_OUTPUT,

    index=False

)


print(
    "Supplementary Macro F1 saved to:"
)


print(
    SUPPLEMENTARY_OUTPUT
)


# %%
# ============================================================
# 47. Final summary
# ============================================================

print(
    "\n"
    + "=" * 100
)


print(
    "XINHUA PRIMARY TOPIC PROMPT VALIDATION COMPLETED"
)


print(
    "=" * 100
)


print(
    "\nNumber of validation cases:",
    len(results)
)


print(
    "\nObserved human topic categories:",
    len(observed_topics)
)


print(
    "\nHuman topic distribution:"
)


display(
    results[
        "human_topic"
    ]
    .value_counts()
)


print(
    "\nAll prompt performance:"
)


display(
    all_evaluation_df.round(4)
)


print(
    "\nP2 vs P2R:"
)


display(
    comparison_df.round(4)
)


print(
    "\nP2R fixed cases:",
    len(fixed)
)


print(
    "P2R worsened cases:",
    len(worsened)
)


print(
    "Net improvement:",
    len(fixed)
    -
    len(worsened)
)


print(
    "\nSupplementary Macro F1:"
)


display(
    supplementary_f1_df.round(4)
)


print(
    "\nMain output files:"
)


print(
    INITIAL_OUTPUT
)


print(
    OUTPUT
)


print(
    DIAGNOSTIC_OUTPUT
)


print(
    SUPPLEMENTARY_OUTPUT
)

Source: Xinhua
Human: Business and Commercialisation
P2-R-CN: Business and Commercialisation
Source: Xinhua
Human: Technology and Innovation
P2-R-CN: Technology and Innovation
Source: Xinhua
Human: Technology and Innovation
P2-R-CN: Technology and Innovation
Source: Xinhua
Human: Business and Commercialisation
P2-R-CN: Business and Commercialisation
Source: Xinhua
Human: Technology and Innovation
P2-R-CN: Technology and Innovation
Processing P2-R-CN 1/116
Processing P2-R-CN 2/116
Processing P2-R-CN 3/116
Processing P2-R-CN 4/116
Processing P2-R-CN 5/116
Processing P2-R-CN 6/116
Processing P2-R-CN 7/116
Processing P2-R-CN 8/116
Processing P2-R-CN 9/116
Processing P2-R-CN 10/116
Processing P2-R-CN 11/116
Processing P2-R-CN 12/116
Processing P2-R-CN 13/116
Processing P2-R-CN 14/116
Processing P2-R-CN 15/116
Processing P2-R-CN 16/116
Processing P2-R-CN 17/116
Processing P2-R-CN 18/116
Processing P2-R-CN 19/116
Processing P2-R-CN 20/116
Processing P2-R-CN 21/116
Processing P2-R-CN 22/116
Pr

,prompt,n,accuracy,macro_f1,macro_precision,macro_recall,cohen_kappa
0,P2_CN,116,0.7759,0.6247,0.7538,0.5931,0.6702
1,P2R_CN,116,0.7500,0.4174,0.4873,0.4257,0.6453



All Chinese prompt comparison:


,prompt,n,accuracy,macro_f1,macro_precision,macro_recall,cohen_kappa
0,P1_CN,116,0.6810,0.4170,0.5014,0.4249,0.5019
1,P2_CN,116,0.7759,0.6247,0.7538,0.5931,0.6702
2,P3_CN,116,0.7672,0.6243,0.7625,0.5857,0.6520
3,P2R_CN,116,0.7500,0.4174,0.4873,0.4257,0.6453



P2R-CN confusion matrix:


...


P2 vs P2R:


,prompt,n,accuracy,macro_f1,macro_precision,macro_recall,cohen_kappa
0,P2_CN,116,0.7759,0.6247,0.7538,0.5931,0.6702
1,P2R_CN,116,0.7500,0.4174,0.4873,0.4257,0.6453



P2R fixed cases: 5
P2R worsened cases: 8
Net improvement: -3

Supplementary Macro F1:


,prompt,observed_category_macro_f1,substantive_topic_macro_f1_excluding_other,number_observed_categories,number_substantive_categories
0,P2_CN,0.6247,0.5711,8,7
1,P2R_CN,0.4174,0.4771,8,7



Main output files:
/Users/yurujia/Desktop/Dissertation Data/China/xinhua_primary_topic_prompt_validation.xlsx
/Users/yurujia/Desktop/Dissertation Data/China/xinhua_primary_topic_prompt_validation_with_P2R.xlsx
/Users/yurujia/Desktop/Dissertation Data/China/xinhua_topic_prompt_detailed_diagnostics_P2R.xlsx
/Users/yurujia/Desktop/Dissertation Data/China/xinhua_topic_prompt_supplementary_macro_f1_P2R.xlsx


In [48]:
# %%
# %%
# ============================================================
# P1R-CN
# Refined Basic Prompt
#
# Refinement based on P1-CN error analysis.
#
# Main observed error:
# systematic over-classification of Technology and Innovation,
# especially for:
# - Mobility and Social Impact
# - Public Acceptance and Trust
# - Policy and Regulation
# - Business and Commercialisation
# ============================================================


def build_prompt_p1r_cn(text):

    return f"""

你正在为一项关于自动驾驶新闻报道的学术研究进行主题分类。

请识别以下新闻文本中占主导地位的单一主要议题。

必须从以下类别中选择且只能选择一个：

Technology and Innovation

Safety and Risk

Policy and Regulation

Business and Commercialisation

Public Acceptance and Trust

Mobility and Social Impact

Environment and Sustainability

Legal and Ethics

Other


请根据文章的核心关注点判断，而不是仅根据是否出现
“自动驾驶”“技术”“研发”“测试”“智能”等词进行分类。


特别注意以下类别边界：


1. Technology and Innovation

只有当文章主要关注自动驾驶技术本身时选择该类别，例如：

- 技术研发；
- 算法、传感器、软件或车辆系统；
- 技术性能；
- 技术能力提升；
- 工程测试或技术验证；
- 技术突破。

自动驾驶技术被用于某项服务、产业发展、政策或社会讨论，
并不自动意味着文章属于 Technology and Innovation。


2. Mobility and Social Impact

如果文章主要关注自动驾驶如何被用于交通出行或载客服务，
例如：

- Robotaxi；
- 无人驾驶出租车；
- 自动驾驶网约车；
- 自动驾驶载客服务；
- 城市出行；
- 交通模式变化；
- 自动驾驶对日常出行或社会生活的影响；

应优先选择 Mobility and Social Impact，

即使文章同时提到了相关技术、测试或企业。


3. Public Acceptance and Trust

如果文章主要关注人们对自动驾驶的：

- 态度；
- 信任；
- 担忧；
- 接受程度；
- 使用意愿；
- 社会认知；

应选择 Public Acceptance and Trust，

即使文本同时讨论了自动驾驶技术。


4. Policy and Regulation

如果文章主要关注：

- 政府政策；
- 法律法规；
- 官方标准；
- 测试许可；
- 政府批准；
- 监管措施；
- 政府推动或限制自动驾驶发展的制度安排；

应选择 Policy and Regulation，

即使政策涉及技术测试或产业发展。


5. Business and Commercialisation

如果文章主要关注：

- 企业战略；
- 投资；
- 商业合作；
- 市场竞争；
- 商业化；
- 产业发展；
- 市场规模；
- 企业布局；

应选择 Business and Commercialisation，

而不是因为其中涉及自动驾驶技术就选择 Technology and Innovation。


6. Safety and Risk

如果文章主要关注事故、碰撞、故障、安全风险、
可靠性或安全评估，

选择 Safety and Risk。


判断时：

先确定“这段新闻最主要在讨论什么问题”，
然后选择最能代表该核心问题的类别。

不要根据孤立关键词分类。

当多个主题同时出现时，
选择在文本中最受强调、最能代表核心新闻框架的一个主题。

不要创建新的类别。

只返回一个上述英文类别名称，不要解释。


新闻文本：

{text}

""".strip()

In [49]:
# %%
# %%
# ============================================================
# 21. Test P1R-CN
# ============================================================


for idx, row in validation.head(5).iterrows():

    print(
        "=" * 80
    )


    print(
        "Source:",
        row["source"]
    )


    print(
        "Human:",
        row["human_topic"]
    )


    prediction = classify_topic(
        build_prompt_p1r_cn(
            row["text"]
        )
    )


    print(
        "P1-R-CN:",
        prediction
    )



Source: Xinhua
Human: Business and Commercialisation
P1-R-CN: Business and Commercialisation
Source: Xinhua
Human: Technology and Innovation
P1-R-CN: Business and Commercialisation
Source: Xinhua
Human: Technology and Innovation
P1-R-CN: Technology and Innovation
Source: Xinhua
Human: Business and Commercialisation
P1-R-CN: Business and Commercialisation
Source: Xinhua
Human: Technology and Innovation
P1-R-CN: Technology and Innovation


In [ ]:
# %%
# %%
# ============================================================
# 22. Run P1R-CN on full validation sample
# ============================================================


results[
    "P1R_CN"
] = None


for idx, row in results.iterrows():

    print(
        f"Processing P1-R-CN "
        f"{idx + 1}/{len(results)}"
    )


    results.at[
        idx,
        "P1R_CN"
    ] = classify_topic(
        build_prompt_p1r_cn(
            row["text"]
        )
    )


print(
    "P1-R-CN completed."
)

Processing P3-R-CN 1/116
Processing P3-R-CN 2/116
Processing P3-R-CN 3/116
Processing P3-R-CN 4/116
Processing P3-R-CN 5/116
Processing P3-R-CN 6/116
Processing P3-R-CN 7/116
Processing P3-R-CN 8/116
Processing P3-R-CN 9/116
Processing P3-R-CN 10/116
Processing P3-R-CN 11/116
Processing P3-R-CN 12/116
Processing P3-R-CN 13/116
Processing P3-R-CN 14/116
Processing P3-R-CN 15/116
Processing P3-R-CN 16/116
Processing P3-R-CN 17/116
Processing P3-R-CN 18/116
Processing P3-R-CN 19/116
Processing P3-R-CN 20/116
Processing P3-R-CN 21/116
Processing P3-R-CN 22/116
Processing P3-R-CN 23/116
Processing P3-R-CN 24/116
Processing P3-R-CN 25/116
Processing P3-R-CN 26/116
Processing P3-R-CN 27/116
Processing P3-R-CN 28/116
Processing P3-R-CN 29/116
Processing P3-R-CN 30/116
Processing P3-R-CN 31/116
Processing P3-R-CN 32/116
Processing P3-R-CN 33/116
Processing P3-R-CN 34/116
Processing P3-R-CN 35/116
Processing P3-R-CN 36/116
Processing P3-R-CN 37/116
Processing P3-R-CN 38/116
Processing P3-R-CN 39

In [51]:
# %%
# %%
# ============================================================
# 23. Validate all model output labels
# ============================================================


for prompt in [

    "P1_CN",

    "P2_CN",

    "P3_CN",

    "P1R_CN"

]:


    if prompt not in results.columns:

        continue


    unexpected = (

        results.loc[
            results[prompt].notna()
            &
            ~results[prompt].isin(TOPICS),
            prompt
        ]
        .value_counts()

    )


    print(
        "\n" + "=" * 60
    )


    print(
        "Checking:",
        prompt
    )


    if len(unexpected) == 0:

        print(
            "All predictions are valid topic labels."
        )

    else:

        print(
            "WARNING: Unexpected model outputs found:"
        )

        display(
            unexpected
        )


# %%
# %%
# ============================================================
# 24. Compare P1-CN and P1R-CN
#
# Same evaluation criteria as English analysis.
# ============================================================


prompt_columns = [

    "P1_CN",

    "P1R_CN"

]


comparison_rows = []


for prompt in prompt_columns:


    valid = results[
        results[prompt].notna()
        &
        results["human_topic"].notna()
        &
        results[prompt].isin(TOPICS)
    ].copy()


    y_true = valid[
        "human_topic"
    ]


    y_pred = valid[
        prompt
    ]


    comparison_rows.append({

        "prompt":
            prompt,

        "n":
            len(valid),

        "accuracy":
            accuracy_score(
                y_true,
                y_pred
            ),

        "macro_precision":
            precision_score(
                y_true,
                y_pred,
                labels=observed_topics,
                average="macro",
                zero_division=0
            ),

        "macro_recall":
            recall_score(
                y_true,
                y_pred,
                labels=observed_topics,
                average="macro",
                zero_division=0
            ),

        "macro_f1":
            f1_score(
                y_true,
                y_pred,
                labels=observed_topics,
                average="macro",
                zero_division=0
            ),

        "cohen_kappa":
            cohen_kappa_score(
                y_true,
                y_pred
            )

    })


comparison_df = pd.DataFrame(
    comparison_rows
)


comparison_df = comparison_df[
    [
        "prompt",
        "n",
        "accuracy",
        "macro_f1",
        "macro_precision",
        "macro_recall",
        "cohen_kappa"
    ]
]


print(
    "\nP1-CN vs P1R-CN:"
)


display(
    comparison_df.round(4)
)



Checking: P1_CN
All predictions are valid topic labels.

Checking: P2_CN
All predictions are valid topic labels.

Checking: P3_CN
All predictions are valid topic labels.

Checking: P1R_CN
All predictions are valid topic labels.

P1-CN vs P1R-CN:


,prompt,n,accuracy,macro_f1,macro_precision,macro_recall,cohen_kappa
0,P1_CN,116,0.6810,0.4170,0.5014,0.4249,0.5019
1,P1R_CN,116,0.6897,0.6178,0.6571,0.6666,0.5882


In [52]:
# %%
# %%
# ============================================================
# 25. Compare all four Chinese prompts
# ============================================================


all_prompt_columns = [

    "P1_CN",

    "P2_CN",

    "P3_CN",

    "P1R_CN"

]


all_evaluation_rows = []


for prompt in all_prompt_columns:


    valid = results[
        results[prompt].notna()
        &
        results["human_topic"].notna()
        &
        results[prompt].isin(TOPICS)
    ].copy()


    y_true = valid[
        "human_topic"
    ]


    y_pred = valid[
        prompt
    ]


    all_evaluation_rows.append({

        "prompt":
            prompt,

        "n":
            len(valid),

        "accuracy":
            accuracy_score(
                y_true,
                y_pred
            ),

        "macro_precision":
            precision_score(
                y_true,
                y_pred,
                labels=observed_topics,
                average="macro",
                zero_division=0
            ),

        "macro_recall":
            recall_score(
                y_true,
                y_pred,
                labels=observed_topics,
                average="macro",
                zero_division=0
            ),

        "macro_f1":
            f1_score(
                y_true,
                y_pred,
                labels=observed_topics,
                average="macro",
                zero_division=0
            ),

        "cohen_kappa":
            cohen_kappa_score(
                y_true,
                y_pred
            )

    })


all_evaluation_df = pd.DataFrame(
    all_evaluation_rows
)


all_evaluation_df = all_evaluation_df[
    [
        "prompt",
        "n",
        "accuracy",
        "macro_f1",
        "macro_precision",
        "macro_recall",
        "cohen_kappa"
    ]
]


print(
    "\nAll Chinese prompt comparison:"
)


display(
    all_evaluation_df.round(4)
)


# %%
# %%
# ============================================================
# 26. Confusion matrix P1-R-CN
# ============================================================


valid_p1r = results[
    results["human_topic"].notna()
    &
    results["P1R_CN"].notna()
    &
    results["P1R_CN"].isin(TOPICS)
].copy()



cm_p1r = confusion_matrix(

    valid_p1r[
        "human_topic"
    ],

    valid_p1r[
        "P1R_CN"
    ],

    labels=TOPICS

)


cm_p1r_df = pd.DataFrame(

    cm_p1r,

    index=TOPICS,

    columns=TOPICS

)


print(
    "\nP1R-CN confusion matrix:"
)


display(
    cm_p1r_df
)


All Chinese prompt comparison:


,prompt,n,accuracy,macro_f1,macro_precision,macro_recall,cohen_kappa
0,P1_CN,116,0.6810,0.4170,0.5014,0.4249,0.5019
1,P2_CN,116,0.7759,0.6247,0.7538,0.5931,0.6702
2,P3_CN,116,0.7672,0.6243,0.7625,0.5857,0.6520
3,P1R_CN,116,0.6897,0.6178,0.6571,0.6666,0.5882



P1R-CN confusion matrix:


,Technology and Innovation,Safety and Risk,Policy and Regulation,Business and Commercialisation,Public Acceptance and Trust,Mobility and Social Impact,Environment and Sustainability,Legal and Ethics,Other
Technology and Innovation,34,0,1,11,0,6,2,0,0
Safety and Risk,0,2,0,0,0,0,0,0,0
Policy and Regulation,1,0,12,3,0,1,0,0,0
Business and Commercialisation,0,0,2,18,0,2,0,0,0
Public Acceptance and Trust,1,0,0,1,2,1,0,0,1
Mobility and Social Impact,1,0,0,1,0,11,0,0,0
Environment and Sustainability,0,0,0,1,0,0,0,0,0
Legal and Ethics,0,0,0,0,0,0,0,0,0
Other,0,0,0,0,0,0,0,0,1


In [53]:
# %%
# %%
# ============================================================
# 27. P1-R-CN errors
# ============================================================


p1r_errors = results[
    results["P1R_CN"].notna()
    &
    (
        results["human_topic"]
        !=
        results["P1R_CN"]
    )
].copy()


print(
    "Number of P1-R-CN errors:",
    len(p1r_errors)
)


display(

    p1r_errors[
        [
            "source",
            "text",
            "human_topic",
            "P1_CN",
            "P1R_CN"
        ]
    ]

)


# %%
# %%
# ============================================================
# 28. Error transitions for P1-R-CN
#
# Shows:
# Human topic -> GPT topic
# ============================================================


p1r_error_transitions = (

    p1r_errors[
        p1r_errors["P1R_CN"].isin(TOPICS)
    ]

    .groupby(
        [
            "human_topic",
            "P1R_CN"
        ]
    )

    .size()

    .reset_index(
        name="count"
    )

    .sort_values(
        "count",
        ascending=False
    )

)


print(
    "\nP1R-CN error transitions:"
)


display(
    p1r_error_transitions
)


Number of P1-R-CN errors: 36


,source,text,human_topic,P1_CN,P1R_CN
11,Xinhua,广汽去年４月已在硅谷建立研发中心，主要开发智能汽车系统、自动驾驶汽车以及其他能源汽车技术。广...,Technology and Innovation,Technology and Innovation,Business and Commercialisation
25,Xinhua,在自动驾驶领域，大众汽车集团正在中国加速发展自动驾驶技术。目前，奥迪品牌已在北京和无锡两座城...,Technology and Innovation,Technology and Innovation,Business and Commercialisation
28,Xinhua,扑面而来的5G（第五代移动通信技术）带来汽车网联化与智能化，中国2018年新能源汽车产销量双...,Technology and Innovation,Mobility and Social Impact,Mobility and Social Impact
29,Xinhua,当天，广汽传祺发布了由广汽洛杉矶前瞻设计中心主导设计的全新概念汽车ENTRANZE，引发全场...,Technology and Innovation,Technology and Innovation,Business and Commercialisation
32,Xinhua,产业通商资源部表示，韩国吸引外国直接投资的重点领域将包括汽车自动驾驶、医疗、智慧家庭、节能方...,Business and Commercialisation,Business and Commercialisation,Policy and Regulation
33,Xinhua,内塔尼亚胡在与斯洛伐克总理佩莱格里尼会晤时表示，以色列方面同意今年晚些时候与斯方在斯洛伐克举...,Business and Commercialisation,Technology and Innovation,Policy and Regulation
37,Xinhua,美国特斯拉汽车公司22日在其位于加利福尼亚州帕洛阿尔托的总部宣布，预计将于2020年第二季度...,Technology and Innovation,Technology and Innovation,Business and Commercialisation
40,Xinhua,王炳南说，与首届进博会相比，第二届进博会新设消费品新品专区，增加养老等题材，新增室外汽车“无...,Public Acceptance and Trust,Business and Commercialisation,Mobility and Social Impact
47,Xinhua,自动避让行人或障碍物，虚线变道超车，与前车保持安全车距、处置复杂突发路况……在湖南省长沙市梅...,Technology and Innovation,Technology and Innovation,Mobility and Social Impact
48,Xinhua,物流运输是钢铁企业控疫保产的重要环节。疫情发生以来，宝钢的“智慧物流”在此也显出了奇效。在6...,Business and Commercialisation,Technology and Innovation,Mobility and Social Impact



P1R-CN error transitions:


,human_topic,P1R_CN,count
12,Technology and Innovation,Business and Commercialisation,11
14,Technology and Innovation,Mobility and Social Impact,6
5,Policy and Regulation,Business and Commercialisation,3
0,Business and Commercialisation,Mobility and Social Impact,2
1,Business and Commercialisation,Policy and Regulation,2
13,Technology and Innovation,Environment and Sustainability,2
2,Environment and Sustainability,Business and Commercialisation,1
3,Mobility and Social Impact,Business and Commercialisation,1
4,Mobility and Social Impact,Technology and Innovation,1
6,Policy and Regulation,Mobility and Social Impact,1


In [54]:


# %%
# %%
# ============================================================
# 29. Cases improved / worsened by P1R-CN
# ============================================================


comparison = results.copy()


comparison["P1_correct"] = (

    comparison["P1_CN"]
    ==
    comparison["human_topic"]

)


comparison["P1R_correct"] = (

    comparison["P1R_CN"]
    ==
    comparison["human_topic"]

)


fixed = comparison[
    (~comparison["P1_correct"])
    &
    (comparison["P1R_correct"])
].copy()


worsened = comparison[
    (comparison["P1_correct"])
    &
    (~comparison["P1R_correct"])
].copy()


print(
    "Fixed by P1-R-CN:",
    len(fixed)
)


print(
    "Worsened by P1-R-CN:",
    len(worsened)
)


print(
    "\nCases fixed by P1-R-CN:"
)


display(
    fixed[
        [
            "text",
            "human_topic",
            "P1_CN",
            "P1R_CN"
        ]
    ]
)


print(
    "\nCases worsened by P1-R-CN:"
)


display(
    worsened[
        [
            "text",
            "human_topic",
            "P1_CN",
            "P1R_CN"
        ]
    ]
)


# %%
# %%
# ============================================================
# 30. P1 vs P1-R summary
# ============================================================


p1_vs_p1r_summary = pd.DataFrame({

    "metric": [

        "Cases fixed by P1R",

        "Cases worsened by P1R",

        "Net improvement"

    ],

    "value": [

        len(fixed),

        len(worsened),

        len(fixed)
        -
        len(worsened)

    ]

})


display(
    p1_vs_p1r_summary
)


# %%
# %%
# ============================================================
# 31. Save P1-R validation results
# ============================================================


OUTPUT = (

    BASE_DIR /

    "xinhua_primary_topic_prompt_validation_with_P1R.xlsx"

)


results.to_excel(

    OUTPUT,

    index=False

)


print(
    "P1-R validation results saved:"
)


print(
    OUTPUT
)


Fixed by P1-R-CN: 21
Worsened by P1-R-CN: 20

Cases fixed by P1-R-CN:


,text,human_topic,P1_CN,P1R_CN
0,王卫东说，中德双方应“共塑创新”，构建双边关系发展新引擎。双方在智能制造、人工智能、自动驾驶...,Business and Commercialisation,Technology and Innovation,Business and Commercialisation
5,“刷脸”与其他创新相结合，应用前景广阔。结合语音技术和无人驾驶技术，可以自动解锁车辆，以自然...,Mobility and Social Impact,Technology and Innovation,Mobility and Social Impact
9,正在中国东部水乡乌镇举行的第四届世界互联网大会描绘了一个令人期待的5G时代：早晨醒来，智能家...,Mobility and Social Impact,Technology and Innovation,Mobility and Social Impact
18,一辆自动驾驶出租车２７日驶上日本首都东京的街头，进行载客试运行。,Mobility and Social Impact,Technology and Innovation,Mobility and Social Impact
21,通用汽车全球执行副总裁兼通用汽车中国公司总裁钱惠康日前曾表示，通用汽车正积极筹备参展首届进口...,Business and Commercialisation,Technology and Innovation,Business and Commercialisation
22,跑完步刷刷脸，就能看到运动数据；坐着无人车，可以逛公园；休息时，可以和亭子“说说话”……来到...,Mobility and Social Impact,Technology and Innovation,Mobility and Social Impact
26,环岛旅游公路除了景色优美，还将很有“智慧”。基于5G技术、GPS定位、大数据、物联网等科技手...,Mobility and Social Impact,Technology and Innovation,Mobility and Social Impact
36,2022年杭州亚运会召开时，亚运区域内将全面实现自动驾驶。这是记者从13日第19届亚运会汽车...,Mobility and Social Impact,Technology and Innovation,Mobility and Social Impact
39,“新能源汽车跨界融合新趋势”是本次大会的主要议题之一。大会设置中重型车零排放论坛、城市交通电...,Mobility and Social Impact,Environment and Sustainability,Mobility and Social Impact
42,周末来公司加班的史先生，用手机下单后，苏宁小店售货员孙荣荣把饮料放到无人车里，设置好开箱密码...,Mobility and Social Impact,Business and Commercialisation,Mobility and Social Impact



Cases worsened by P1-R-CN:


,text,human_topic,P1_CN,P1R_CN
11,广汽去年４月已在硅谷建立研发中心，主要开发智能汽车系统、自动驾驶汽车以及其他能源汽车技术。广...,Technology and Innovation,Technology and Innovation,Business and Commercialisation
25,在自动驾驶领域，大众汽车集团正在中国加速发展自动驾驶技术。目前，奥迪品牌已在北京和无锡两座城...,Technology and Innovation,Technology and Innovation,Business and Commercialisation
29,当天，广汽传祺发布了由广汽洛杉矶前瞻设计中心主导设计的全新概念汽车ENTRANZE，引发全场...,Technology and Innovation,Technology and Innovation,Business and Commercialisation
32,产业通商资源部表示，韩国吸引外国直接投资的重点领域将包括汽车自动驾驶、医疗、智慧家庭、节能方...,Business and Commercialisation,Business and Commercialisation,Policy and Regulation
37,美国特斯拉汽车公司22日在其位于加利福尼亚州帕洛阿尔托的总部宣布，预计将于2020年第二季度...,Technology and Innovation,Technology and Innovation,Business and Commercialisation
47,自动避让行人或障碍物，虚线变道超车，与前车保持安全车距、处置复杂突发路况……在湖南省长沙市梅...,Technology and Innovation,Technology and Innovation,Mobility and Social Impact
50,在抢收小麦的同时，夏管、夏播随即展开。各地正陆续开展“三夏”全程机械化生产现场演示、田间日等...,Technology and Innovation,Technology and Innovation,Environment and Sustainability
52,大湾区5G产业联盟创会会长、中国移动香港有限公司董事兼行政总裁李帆风表示，粤港澳大湾区拥有制...,Technology and Innovation,Technology and Innovation,Business and Commercialisation
53,“中国处于全球5G开发和应用的前沿。工业自动化、智慧城市、自动驾驶等5G技术支持的场景，最终...,Technology and Innovation,Technology and Innovation,Business and Commercialisation
54,3D扫描定制西服、无人驾驶汽车驶上公路、智能生产车间……在中国重庆举行的中国国际智能产业博览...,Technology and Innovation,Technology and Innovation,Business and Commercialisation


,metric,value
0,Cases fixed by P1R,21
1,Cases worsened by P1R,20
2,Net improvement,1


P1-R validation results saved:
/Users/yurujia/Desktop/Dissertation Data/China/xinhua_primary_topic_prompt_validation_with_P1R.xlsx


In [55]:


# %%
# %%
# ============================================================
# 32. Per-topic classification performance
#
# Purpose:
#
# Examine Precision / Recall / F1 / Support
# for each topic for P1_CN and P1R_CN.
# ============================================================


from sklearn.metrics import classification_report


# ------------------------------------------------------------
# Helper function
# ------------------------------------------------------------


def get_per_topic_metrics(
    df,
    prompt_col,
    topic_labels
):


    valid = df[
        df[prompt_col].notna()
        &
        df["human_topic"].notna()
        &
        df[prompt_col].isin(TOPICS)
    ].copy()


    y_true = valid[
        "human_topic"
    ]


    y_pred = valid[
        prompt_col
    ]


    report = classification_report(

        y_true,

        y_pred,

        labels=topic_labels,

        target_names=topic_labels,

        output_dict=True,

        zero_division=0

    )


    rows = []


    for topic in topic_labels:


        metrics = report[
            topic
        ]


        rows.append({

            "prompt":
                prompt_col,

            "topic":
                topic,

            "precision":
                metrics[
                    "precision"
                ],

            "recall":
                metrics[
                    "recall"
                ],

            "f1_score":
                metrics[
                    "f1-score"
                ],

            "support":
                int(
                    metrics[
                        "support"
                    ]
                )

        })


    return pd.DataFrame(
        rows
    )


# %%
# %%
# ============================================================
# 33. Per-topic metrics for P1-CN
# ============================================================


p1_per_topic = get_per_topic_metrics(

    results,

    "P1_CN",

    TOPICS

)


print(
    "\nP1-CN per-topic metrics:"
)


display(
    p1_per_topic.round(4)
)


# %%
# %%
# ============================================================
# 34. Per-topic metrics for P1R-CN
# ============================================================


p1r_per_topic = get_per_topic_metrics(

    results,

    "P1R_CN",

    TOPICS

)


print(
    "\nP1R-CN per-topic metrics:"
)


display(
    p1r_per_topic.round(4)
)


# %%
# %%
# ============================================================
# 35. Combine P1 and P1-R per-topic metrics
# ============================================================


per_topic_comparison = pd.concat(

    [

        p1_per_topic,

        p1r_per_topic

    ],

    ignore_index=True

)


display(
    per_topic_comparison.round(4)
)



P1-CN per-topic metrics:


,prompt,topic,precision,recall,f1_score,support
0,P1_CN,Technology and Innovation,0.6623,0.9444,0.7786,54
1,P1_CN,Safety and Risk,0.6667,1.0000,0.8000,2
2,P1_CN,Policy and Regulation,1.0000,0.5294,0.6923,17
3,P1_CN,Business and Commercialisation,0.6818,0.6818,0.6818,22
4,P1_CN,Public Acceptance and Trust,0.5000,0.1667,0.2500,6
5,P1_CN,Mobility and Social Impact,0.5000,0.0769,0.1333,13
6,P1_CN,Environment and Sustainability,0.0000,0.0000,0.0000,1
7,P1_CN,Legal and Ethics,0.0000,0.0000,0.0000,0
8,P1_CN,Other,0.0000,0.0000,0.0000,1



P1R-CN per-topic metrics:


,prompt,topic,precision,recall,f1_score,support
0,P1R_CN,Technology and Innovation,0.9189,0.6296,0.7473,54
1,P1R_CN,Safety and Risk,1.0000,1.0000,1.0000,2
2,P1R_CN,Policy and Regulation,0.8000,0.7059,0.7500,17
3,P1R_CN,Business and Commercialisation,0.5143,0.8182,0.6316,22
4,P1R_CN,Public Acceptance and Trust,1.0000,0.3333,0.5000,6
5,P1R_CN,Mobility and Social Impact,0.5238,0.8462,0.6471,13
6,P1R_CN,Environment and Sustainability,0.0000,0.0000,0.0000,1
7,P1R_CN,Legal and Ethics,0.0000,0.0000,0.0000,0
8,P1R_CN,Other,0.5000,1.0000,0.6667,1


,prompt,topic,precision,recall,f1_score,support
0,P1_CN,Technology and Innovation,0.6623,0.9444,0.7786,54
1,P1_CN,Safety and Risk,0.6667,1.0000,0.8000,2
2,P1_CN,Policy and Regulation,1.0000,0.5294,0.6923,17
3,P1_CN,Business and Commercialisation,0.6818,0.6818,0.6818,22
4,P1_CN,Public Acceptance and Trust,0.5000,0.1667,0.2500,6
5,P1_CN,Mobility and Social Impact,0.5000,0.0769,0.1333,13
6,P1_CN,Environment and Sustainability,0.0000,0.0000,0.0000,1
7,P1_CN,Legal and Ethics,0.0000,0.0000,0.0000,0
8,P1_CN,Other,0.0000,0.0000,0.0000,1
9,P1R_CN,Technology and Innovation,0.9189,0.6296,0.7473,54


In [56]:


# %%
# %%
# ============================================================
# 36. Human topic support
#
# Important because low-support topics can have unstable F1.
# ============================================================


topic_support = (

    results[
        "human_topic"
    ]

    .value_counts()

    .reindex(
        TOPICS,
        fill_value=0
    )

    .reset_index()

)


topic_support.columns = [

    "topic",

    "human_support"

]


print(
    "\nHuman topic support:"
)


display(
    topic_support
)


# %%
# %%
# ============================================================
# 36. Per-topic metrics for P1-CN
# ============================================================


p1_per_topic = get_per_topic_metrics(

    results,

    "P1_CN",

    TOPICS

)


print(
    "\nP1-CN per-topic metrics:"
)


display(
    p1_per_topic.round(4)
)



Human topic support:


,topic,human_support
0,Technology and Innovation,54
1,Safety and Risk,2
2,Policy and Regulation,17
3,Business and Commercialisation,22
4,Public Acceptance and Trust,6
5,Mobility and Social Impact,13
6,Environment and Sustainability,1
7,Legal and Ethics,0
8,Other,1



P1-CN per-topic metrics:


,prompt,topic,precision,recall,f1_score,support
0,P1_CN,Technology and Innovation,0.6623,0.9444,0.7786,54
1,P1_CN,Safety and Risk,0.6667,1.0000,0.8000,2
2,P1_CN,Policy and Regulation,1.0000,0.5294,0.6923,17
3,P1_CN,Business and Commercialisation,0.6818,0.6818,0.6818,22
4,P1_CN,Public Acceptance and Trust,0.5000,0.1667,0.2500,6
5,P1_CN,Mobility and Social Impact,0.5000,0.0769,0.1333,13
6,P1_CN,Environment and Sustainability,0.0000,0.0000,0.0000,1
7,P1_CN,Legal and Ethics,0.0000,0.0000,0.0000,0
8,P1_CN,Other,0.0000,0.0000,0.0000,1


In [57]:


# %%
# %%
# ============================================================
# 37. Normalised confusion matrix for P1-CN
#
# Each row sums to 1 for categories with observations.
#
# Shows where each human topic is being misclassified.
# ============================================================


valid_p1 = results[
    results["human_topic"].notna()
    &
    results["P1_CN"].notna()
    &
    results["P1_CN"].isin(TOPICS)
].copy()


cm_p1_normalised = confusion_matrix(

    valid_p1[
        "human_topic"
    ],

    valid_p1[
        "P1_CN"
    ],

    labels=TOPICS,

    normalize="true"

)



cm_p1_normalised_df = pd.DataFrame(

    cm_p1_normalised,

    index=TOPICS,

    columns=TOPICS

)


print(
    "\nP1-CN normalised confusion matrix:"
)


display(
    cm_p1_normalised_df.round(3)
)


# %%
# %%
# ============================================================
# 38. Normalised confusion matrix for P1R-CN
# ============================================================


cm_p1r_normalised = confusion_matrix(

    valid_p1r[
        "human_topic"
    ],

    valid_p1r[
        "P1R_CN"
    ],

    labels=TOPICS,

    normalize="true"

)


cm_p1r_normalised_df = pd.DataFrame(

    cm_p1r_normalised,

    index=TOPICS,

    columns=TOPICS

)


print(
    "\nP1R-CN normalised confusion matrix:"
)


display(
    cm_p1r_normalised_df.round(3)
)



P1-CN normalised confusion matrix:


,Technology and Innovation,Safety and Risk,Policy and Regulation,Business and Commercialisation,Public Acceptance and Trust,Mobility and Social Impact,Environment and Sustainability,Legal and Ethics,Other
Technology and Innovation,0.944,0.019,0.000,0.019,0.000,0.019,0.000,0.0,0.0
Safety and Risk,0.000,1.000,0.000,0.000,0.000,0.000,0.000,0.0,0.0
Policy and Regulation,0.353,0.000,0.529,0.118,0.000,0.000,0.000,0.0,0.0
Business and Commercialisation,0.318,0.000,0.000,0.682,0.000,0.000,0.000,0.0,0.0
Public Acceptance and Trust,0.500,0.000,0.000,0.333,0.167,0.000,0.000,0.0,0.0
Mobility and Social Impact,0.769,0.000,0.000,0.077,0.000,0.077,0.077,0.0,0.0
Environment and Sustainability,0.000,0.000,0.000,1.000,0.000,0.000,0.000,0.0,0.0
Legal and Ethics,0.000,0.000,0.000,0.000,0.000,0.000,0.000,0.0,0.0
Other,0.000,0.000,0.000,0.000,1.000,0.000,0.000,0.0,0.0



P1R-CN normalised confusion matrix:


,Technology and Innovation,Safety and Risk,Policy and Regulation,Business and Commercialisation,Public Acceptance and Trust,Mobility and Social Impact,Environment and Sustainability,Legal and Ethics,Other
Technology and Innovation,0.630,0.0,0.019,0.204,0.000,0.111,0.037,0.0,0.000
Safety and Risk,0.000,1.0,0.000,0.000,0.000,0.000,0.000,0.0,0.000
Policy and Regulation,0.059,0.0,0.706,0.176,0.000,0.059,0.000,0.0,0.000
Business and Commercialisation,0.000,0.0,0.091,0.818,0.000,0.091,0.000,0.0,0.000
Public Acceptance and Trust,0.167,0.0,0.000,0.167,0.333,0.167,0.000,0.0,0.167
Mobility and Social Impact,0.077,0.0,0.000,0.077,0.000,0.846,0.000,0.0,0.000
Environment and Sustainability,0.000,0.0,0.000,1.000,0.000,0.000,0.000,0.0,0.000
Legal and Ethics,0.000,0.0,0.000,0.000,0.000,0.000,0.000,0.0,0.000
Other,0.000,0.0,0.000,0.000,0.000,0.000,0.000,0.0,1.000


In [58]:


# %%
# %%
# ============================================================
# 39. Identify weak-performing topics
#
# Threshold is diagnostic only.
#
# F1 < 0.50 is NOT treated as a universal statistical cutoff.
# ============================================================



weak_topics_p1 = p1_per_topic[
    p1_per_topic[
        "f1_score"
    ] < 0.50
].copy()


print(
    "P1_CN topics with F1 < 0.50:"
)


display(
    weak_topics_p1.round(4)
)



weak_topics_p1r = p1r_per_topic[
    p1r_per_topic[
        "f1_score"
    ] < 0.50
].copy()


print(
    "\nP1R_CN topics with F1 < 0.50:"
)


display(
    weak_topics_p1r.round(4)
)


# %%
# %%
# ============================================================
# 40. Compare per-topic F1 change from P1 to P1-R
# ============================================================


f1_change = (

    p1_per_topic[
        [
            "topic",
            "f1_score"
        ]
    ]

    .rename(
        columns={
            "f1_score":
                "P1_F1"
        }
    )

    .merge(

        p1r_per_topic[
            [
                "topic",
                "f1_score"
            ]
        ]

        .rename(
            columns={
                "f1_score":
                    "P1R_F1"
            }
        ),

        on="topic",

        how="outer"

    )

)


f1_change[
    "F1_change_P1R_minus_P1"
] = (

    f1_change[
        "P1R_F1"
    ]

    -

    f1_change[
        "P1_F1"
    ]

)


print(
    "\nPer-topic F1 change:"
)


display(
    f1_change.round(4)
)


P1_CN topics with F1 < 0.50:


,prompt,topic,precision,recall,f1_score,support
4,P1_CN,Public Acceptance and Trust,0.5,0.1667,0.2500,6
5,P1_CN,Mobility and Social Impact,0.5,0.0769,0.1333,13
6,P1_CN,Environment and Sustainability,0.0,0.0000,0.0000,1
7,P1_CN,Legal and Ethics,0.0,0.0000,0.0000,0
8,P1_CN,Other,0.0,0.0000,0.0000,1



P1R_CN topics with F1 < 0.50:


,prompt,topic,precision,recall,f1_score,support
6,P1R_CN,Environment and Sustainability,0.0,0.0,0.0,1
7,P1R_CN,Legal and Ethics,0.0,0.0,0.0,0



Per-topic F1 change:


,topic,P1_F1,P1R_F1,F1_change_P1R_minus_P1
0,Business and Commercialisation,0.6818,0.6316,-0.0502
1,Environment and Sustainability,0.0000,0.0000,0.0000
2,Legal and Ethics,0.0000,0.0000,0.0000
3,Mobility and Social Impact,0.1333,0.6471,0.5137
4,Other,0.0000,0.6667,0.6667
5,Policy and Regulation,0.6923,0.7500,0.0577
6,Public Acceptance and Trust,0.2500,0.5000,0.2500
7,Safety and Risk,0.8000,1.0000,0.2000
8,Technology and Innovation,0.7786,0.7473,-0.0314


In [65]:


# %%
# %%
# ============================================================
# 41. Label distribution comparison
#
# Human vs P1 / P2 / P3 / P1R
#
# Useful for checking whether a prompt systematically
# over-predicts a particular category.
# ============================================================


distribution_columns = [

    "human_topic",

    "P1_CN",

    "P2_CN",

    "P3_CN",

    "P1R_CN"

]


distribution_dict = {}


for col in distribution_columns:


    counts = (

        results[col]

        .dropna()

        .value_counts()

        .reindex(
            TOPICS,
            fill_value=0
        )

    )


    distribution_dict[
        col
    ] = counts


topic_distribution_df = pd.DataFrame(
    distribution_dict
)


print(
    "\nTopic label counts:"
)


display(
    topic_distribution_df
)


# %%
# %%
# ============================================================
# 42. Topic distribution percentages
# ============================================================


topic_distribution_pct_df = (

    topic_distribution_df

    /

    topic_distribution_df.sum(
        axis=0
    )

    *

    100

)


print(
    "\nTopic label percentages:"
)


display(
    topic_distribution_pct_df.round(2)
)


# %%
# %%
# ============================================================
# 43. Save detailed diagnostic results
# ============================================================


DIAGNOSTIC_OUTPUT = (

    BASE_DIR /

    "xinhua_topic_prompt_detailed_diagnostics.xlsx"

)


with pd.ExcelWriter(

    DIAGNOSTIC_OUTPUT,

    engine="openpyxl"

) as writer:


    # ---------------------------------------
    # Article-level predictions
    # ---------------------------------------

    results.to_excel(

        writer,

        sheet_name="article_results",

        index=False

    )


    # ---------------------------------------
    # All prompt overall metrics
    # ---------------------------------------

    all_evaluation_df.to_excel(

        writer,

        sheet_name="all_prompt_metrics",

        index=False

    )


    # ---------------------------------------
    # P1 vs P1R metrics
    # ---------------------------------------

    comparison_df.to_excel(

        writer,

        sheet_name="P1_vs_P1R_metrics",

        index=False

    )


    # ---------------------------------------
    # P1 per-topic metrics
    # ---------------------------------------

    p1_per_topic.to_excel(

        writer,

        sheet_name="P1_per_topic",

        index=False

    )


    # ---------------------------------------
    # P1R per-topic metrics
    # ---------------------------------------

    p1r_per_topic.to_excel(

        writer,

        sheet_name="P1R_per_topic",

        index=False

    )


    # ---------------------------------------
    # F1 comparison
    # ---------------------------------------

    f1_change.to_excel(

        writer,

        sheet_name="F1_comparison",

        index=False

    )


    # ---------------------------------------
    # Human topic support
    # ---------------------------------------

    topic_support.to_excel(

        writer,

        sheet_name="topic_support",

        index=False

    )


    # ---------------------------------------
    # P1 raw confusion matrix
    # ---------------------------------------

    cm_p1_df.to_excel(

        writer,

        sheet_name="P1_confusion_raw"

    )


    # ---------------------------------------
    # P1 normalised confusion matrix
    # ---------------------------------------

    cm_p1_normalised_df.to_excel(

        writer,

        sheet_name="P1_confusion_normalised"

    )


    # ---------------------------------------
    # P1R raw confusion matrix
    # ---------------------------------------

    cm_p1r_df.to_excel(

        writer,

        sheet_name="P1R_confusion_raw"

    )


    # ---------------------------------------
    # P1R normalised confusion matrix
    # ---------------------------------------

    cm_p1r_normalised_df.to_excel(

        writer,

        sheet_name="P1R_confusion_normalised"

    )


    # ---------------------------------------
    # P1 weak topics
    # ---------------------------------------

    weak_topics_p1.to_excel(

        writer,

        sheet_name="P1_weak_topics",

        index=False

    )


    # ---------------------------------------
    # P1R weak topics
    # ---------------------------------------

    weak_topics_p1r.to_excel(

        writer,

        sheet_name="P1R_weak_topics",

        index=False

    )


    # ---------------------------------------
    # P1R errors
    # ---------------------------------------

    p1r_errors.to_excel(

        writer,

        sheet_name="P1R_errors",

        index=False

    )


    # ---------------------------------------
    # P1R error transitions
    # ---------------------------------------

    p1r_error_transitions.to_excel(

        writer,

        sheet_name="P1R_error_transitions",

        index=False

    )


    # ---------------------------------------
    # Cases fixed by P1R
    # ---------------------------------------

    fixed.to_excel(

        writer,

        sheet_name="fixed_by_P1R",

        index=False

    )


    # ---------------------------------------
    # Cases worsened by P1R
    # ---------------------------------------

    worsened.to_excel(

        writer,

        sheet_name="worsened_by_P1R",

        index=False

    )


    # ---------------------------------------
    # P1 vs P1R summary
    # ---------------------------------------

    p1_vs_p1r_summary.to_excel(

        writer,

        sheet_name="P1_vs_P1R_summary",

        index=False

    )


    # ---------------------------------------
    # Topic counts
    # ---------------------------------------

    topic_distribution_df.to_excel(

        writer,

        sheet_name="topic_counts"

    )


    # ---------------------------------------
    # Topic percentages
    # ---------------------------------------

    topic_distribution_pct_df.to_excel(

        writer,

        sheet_name="topic_percentages"

    )


print(
    "Detailed diagnostics saved to:"
)


print(
    DIAGNOSTIC_OUTPUT
)




Topic label counts:


,human_topic,P1_CN,P2_CN,P3_CN,P1R_CN
Technology and Innovation,54,77,63,67,37
Safety and Risk,2,3,1,1,2
Policy and Regulation,17,9,15,15,15
Business and Commercialisation,22,22,21,19,35
Public Acceptance and Trust,6,2,1,1,2
Mobility and Social Impact,13,2,14,12,21
Environment and Sustainability,1,1,0,0,2
Legal and Ethics,0,0,0,0,0
Other,1,0,1,1,2



Topic label percentages:


,human_topic,P1_CN,P2_CN,P3_CN,P1R_CN
Technology and Innovation,46.55,66.38,54.31,57.76,31.90
Safety and Risk,1.72,2.59,0.86,0.86,1.72
Policy and Regulation,14.66,7.76,12.93,12.93,12.93
Business and Commercialisation,18.97,18.97,18.10,16.38,30.17
Public Acceptance and Trust,5.17,1.72,0.86,0.86,1.72
Mobility and Social Impact,11.21,1.72,12.07,10.34,18.10
Environment and Sustainability,0.86,0.86,0.00,0.00,1.72
Legal and Ethics,0.00,0.00,0.00,0.00,0.00
Other,0.86,0.00,0.86,0.86,1.72


Detailed diagnostics saved to:
/Users/yurujia/Desktop/Dissertation Data/China/xinhua_topic_prompt_detailed_diagnostics.xlsx


In [66]:

# %%
# %%
# ============================================================
# 44. Supplementary Macro F1 diagnostics
#
# 1. Observed-category Macro F1:
#    Includes all categories represented in human validation,
#    including "Other".
#
# 2. Substantive-topic Macro F1:
#    Excludes the residual "Other" category.
#
# The second measure is supplementary only.
#
# Main dissertation results should still report the
# observed-category Macro F1 used in the main evaluation table.
#
# Same logic as English validation.
# ============================================================


substantive_observed_topics = [

    topic

    for topic in observed_topics

    if topic != "Other"

]


print(
    "Observed categories:"
)


print(
    observed_topics
)


print(
    "\nSubstantive observed categories "
    "(excluding Other):"
)


print(
    substantive_observed_topics
)


# %%
# %%
# ============================================================
# 45. Supplementary Macro F1 for P1 and P1R
# ============================================================


supplementary_rows = []


for prompt in [

    "P1_CN",

    "P1R_CN"

]:


    valid = results[
        results[prompt].notna()
        &
        results["human_topic"].notna()
        &
        results[prompt].isin(TOPICS)
    ].copy()


    y_true = valid[
        "human_topic"
    ]


    y_pred = valid[
        prompt
    ]


    observed_macro_f1 = f1_score(

        y_true,

        y_pred,

        labels=observed_topics,

        average="macro",

        zero_division=0

    )


    substantive_macro_f1 = f1_score(

        y_true,

        y_pred,

        labels=substantive_observed_topics,

        average="macro",

        zero_division=0

    )


    supplementary_rows.append({

        "prompt":
            prompt,

        "observed_category_macro_f1":
            observed_macro_f1,

        "substantive_topic_macro_f1_excluding_other":
            substantive_macro_f1,

        "number_observed_categories":
            len(
                observed_topics
            ),

        "number_substantive_categories":
            len(
                substantive_observed_topics
            )

    })


supplementary_f1_df = pd.DataFrame(
    supplementary_rows
)


print(
    "\nSupplementary Macro F1 diagnostics:"
)


display(
    supplementary_f1_df.round(4)
)


# %%
# %%
# ============================================================
# 46. Save supplementary Macro F1 diagnostics
# ============================================================


SUPPLEMENTARY_OUTPUT = (

    BASE_DIR /

    "xinhua_topic_prompt_supplementary_macro_f1.xlsx"

)


supplementary_f1_df.to_excel(

    SUPPLEMENTARY_OUTPUT,

    index=False

)


print(
    "Supplementary Macro F1 saved to:"
)


print(
    SUPPLEMENTARY_OUTPUT
)


Observed categories:
['Technology and Innovation', 'Safety and Risk', 'Policy and Regulation', 'Business and Commercialisation', 'Public Acceptance and Trust', 'Mobility and Social Impact', 'Environment and Sustainability', 'Other']

Substantive observed categories (excluding Other):
['Technology and Innovation', 'Safety and Risk', 'Policy and Regulation', 'Business and Commercialisation', 'Public Acceptance and Trust', 'Mobility and Social Impact', 'Environment and Sustainability']

Supplementary Macro F1 diagnostics:


,prompt,observed_category_macro_f1,substantive_topic_macro_f1_excluding_other,number_observed_categories,number_substantive_categories
0,P1_CN,0.4170,0.4766,8,7
1,P1R_CN,0.6178,0.6108,8,7


Supplementary Macro F1 saved to:
/Users/yurujia/Desktop/Dissertation Data/China/xinhua_topic_prompt_supplementary_macro_f1.xlsx


In [67]:


# %%
# %%
# ============================================================
# 47. Final summary
# ============================================================


print(
    "\n"
    + "=" * 100
)


print(
    "XINHUA PRIMARY TOPIC PROMPT VALIDATION COMPLETED"
)


print(
    "=" * 100
)


print(
    "\nNumber of validation cases:",
    len(results)
)


print(
    "\nObserved human topic categories:",
    len(observed_topics)
)


print(
    "\nHuman topic distribution:"
)


display(
    results[
        "human_topic"
    ]
    .value_counts()
)


print(
    "\nAll prompt performance:"
)


display(
    all_evaluation_df.round(4)
)


print(
    "\nP1 vs P1R:"
)


display(
    comparison_df.round(4)
)


print(
    "\nP1R fixed cases:",
    len(fixed)
)


print(
    "P1R worsened cases:",
    len(worsened)
)


print(
    "Net improvement:",
    len(fixed)
    -
    len(worsened)
)


print(
    "\nSupplementary Macro F1:"
)


display(
    supplementary_f1_df.round(4)
)


print(
    "\nMain output files:"
)


print(
    INITIAL_OUTPUT
)


print(
    OUTPUT
)


print(
    DIAGNOSTIC_OUTPUT
)


print(
    SUPPLEMENTARY_OUTPUT
)


XINHUA PRIMARY TOPIC PROMPT VALIDATION COMPLETED

Number of validation cases: 116

Observed human topic categories: 8

Human topic distribution:


human_topic
Technology and Innovation         54
Business and Commercialisation    22
Policy and Regulation             17
Mobility and Social Impact        13
Public Acceptance and Trust        6
Safety and Risk                    2
Other                              1
Environment and Sustainability     1
Name: count, dtype: int64


All prompt performance:


,prompt,n,accuracy,macro_f1,macro_precision,macro_recall,cohen_kappa
0,P1_CN,116,0.6810,0.4170,0.5014,0.4249,0.5019
1,P2_CN,116,0.7759,0.6247,0.7538,0.5931,0.6702
2,P3_CN,116,0.7672,0.6243,0.7625,0.5857,0.6520
3,P1R_CN,116,0.6897,0.6178,0.6571,0.6666,0.5882



P1 vs P1R:


,prompt,n,accuracy,macro_f1,macro_precision,macro_recall,cohen_kappa
0,P1_CN,116,0.6810,0.4170,0.5014,0.4249,0.5019
1,P1R_CN,116,0.6897,0.6178,0.6571,0.6666,0.5882



P1R fixed cases: 21
P1R worsened cases: 20
Net improvement: 1

Supplementary Macro F1:


,prompt,observed_category_macro_f1,substantive_topic_macro_f1_excluding_other,number_observed_categories,number_substantive_categories
0,P1_CN,0.4170,0.4766,8,7
1,P1R_CN,0.6178,0.6108,8,7



Main output files:
/Users/yurujia/Desktop/Dissertation Data/China/xinhua_primary_topic_prompt_validation.xlsx
/Users/yurujia/Desktop/Dissertation Data/China/xinhua_primary_topic_prompt_validation_with_P1R.xlsx
/Users/yurujia/Desktop/Dissertation Data/China/xinhua_topic_prompt_detailed_diagnostics.xlsx
/Users/yurujia/Desktop/Dissertation Data/China/xinhua_topic_prompt_supplementary_macro_f1.xlsx
